# RSNA Knee: 31 Experiments and Honest OOF Analysis

**A controlled experiment journal — 2026-10-19 snapshot**

This notebook reports our own **Gold58 development out-of-fold (OOF) results**, not a 0.95 leaderboard solution. It covers weak report labels, frozen image features, slice sampling, positional encoding/RoPE, finding-specific attention, limited fine-tuning, and ensembling. All figures below are regenerated on CPU from aggregate statistics. The model example uses synthetic tensors, not MRI data.

이 실험의 질문은 **MRI의 정보가 어느 단계에서 사라지고, 어떤 변경이 실제로 12개 소견 예측에 도움이 되는가?** 입니다. 성공한 변경뿐 아니라 성능이 낮아진 실험도 기록합니다. 현재 snapshot의 최고 개발 OOF는 R01 **0.771432 ± 0.008916 (seed SD)**입니다. 이 값과 다른 사람의 public LB 0.95x는 서로 다른 평가이므로 직접 격차로 해석하지 않습니다.

이 Notebook은 방법론·오류 분석용 Code입니다. 새 GPU 학습이나 hidden-test 예측을 실행하지 않으며 `submission.csv`를 만들지 않습니다. 원 MRI, 보고서 원문, study 식별자, 환자별 예측, 타인의 모델 가중치를 배포하지 않습니다.

## 1. Task, targets and evaluation

한 무릎 MRI study에는 여러 series와 slice가 있습니다. 목표는 다음 **12개 소견 각각의 점수**를 출력하는 것입니다.

`ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture`

여러 소견이 동시에 존재할 수 있는 **multi-label classification**입니다. 12개 중 하나를 고르는 softmax/MNIST 형태가 아니라, 12개 logit에 각각 sigmoid를 적용합니다. 정답은 study 단위이며 앞쪽 slice의 정답 이미지와 뒤쪽 test 이미지를 짝지어 비교하는 문제가 아닙니다.

학습은 소견별 BCEWithLogitsLoss를 기반으로 합니다. 알려진 weak label에는 confidence 가중치, gold에는 별도 가중치, UNK에는 0 mask를 적용합니다. 현재 known weak 가중치는 `0.25 + 0.75 × confidence`, gold 가중치는 3입니다. 평가 BCE는 gold의 unweighted 값이며 학습 loss의 절댓값과 직접 비교하지 않습니다.

대회 점수는 12개 소견별 ROC-AUC의 평균입니다. **AUC 0.95는 정확도 95%가 아닙니다.** 한 소견의 양성 study가 음성 study보다 높은 점수를 받는 순위를 측정합니다. 아래 혼동행렬은 별도로 threshold=0.5를 사용합니다. [Competition evaluation](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/overview/evaluation)

## 2. Validation contract and weak labels

- Gold 58개 study, 동일한 3-fold 체계, seed 42/2026/3407에서 OOF 예측을 평가했습니다. 표의 AUC는 seed별 전체 OOF AUC를 구한 뒤 평균한 값입니다. seed SD는 세 값의 변동이며 신뢰구간이 아닙니다.
- Gold만 쓰는 대조군과, 필터링한 report weak 4,292개를 추가하는 실험을 비교했습니다. 후자는 gold까지 총 4,350개 학습 후보입니다. 각 gold validation fold는 해당 모델의 gradient 학습에서 제외합니다.
- 약라벨은 판독 보고서를 해석해 만든 soft target입니다. 영상별로 전문가가 다시 확정한 정답과 같지 않습니다. 알려지지 않았다는 UNK는 정상이라는 NO와 구분합니다. 공개 teacher의 선택 과정과 gold에 대한 독립성 전체를 증명한 것은 아닙니다.
- 58개를 반복적으로 개발·선택에 사용했습니다. 따라서 이 OOF도 완전히 손대지 않은 최종 test가 아닙니다. 세 seed가 독립 환자 174명을 의미하지 않습니다. 외부 public 모델이 gold로 checkpoint를 골랐다면 그 gold 성적과도 평가 조건이 다릅니다.
- 배포용 모델은 full-fit seed42입니다. 세 seed/fold 평가를 했다는 사실이 배포 시 모든 fold/seed를 앙상블했다는 뜻은 아닙니다.

공개 약라벨 출처: [Pilkwang report labels](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels). 이 노트에는 해당 label table이나 보고서를 포함하지 않습니다.

## 3. Our current image-to-prediction pipeline

```text
DICOM study
  → 3 planes × 2 fat-suppression slots (up to 6)
  → order slices by ImagePositionPatient · cross(ImageOrientationPatient)
  → choose centres (default 25–75%) and neighbouring slices
  → deduplicate single-slice tokens; retain order and relative mm coordinates
  → intensity normalization, resize 232, centre crop 224
  → ImageNet Swin-T image encoder (mostly frozen)
  → global-pooled 768-d image feature
  → training-fold mean/std normalization → linear projection to 128-d
  → optional plane/protocol embeddings and within-slot slice self-attention/RoPE
  → class-specific slot pooling + gate (S10)
       OR 12 learned class queries cross-attending to all slice tokens (S11)
  → 12 logits → sigmoid → 12 independent scores
```

E00/E01은 ResNet18 대조군, E02에서 backbone 후보를 비교했습니다. 현재 주력은 Swin-T이며, Swin 자체의 이미지 내부 attention과 그 뒤에 추가한 **study/slice attention은 다른 단계**입니다.

S02의 `5`와 S03의 `10`은 각 중심 주변의 이웃 폭입니다. study 전체가 5장/10장이라는 뜻이 아닙니다. 겹치는 slice를 중복 제거하므로 token 수도 정확히 두 배가 되지 않습니다. S00/E08의 triplet은 인접 3장을 RGB 세 채널처럼 한 encoder 입력으로 넣는 방식이고, S01–S03은 각 slice를 별도 token으로 만드는 방식입니다.

**중요한 한계:** 현재 cross-attention 앞에는 이미 이미지 global pooling과 768→128 투영이 있습니다. cross-attention이 원래 사라진 patch 정보를 복원해 주지는 않습니다. 또한 mm RoPE를 쓴다고 PixelSpacing에 맞는 물리적 이미지 crop을 적용한 것은 아닙니다. 그 두 변경은 다음 실험에서 분리합니다.

In [ ]:
import io, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display

OUT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
torch.set_num_threads(2)
print('Snapshot: 2026-10-19; CPU analysis; no patient-level records')

In [ ]:
stats = pd.read_json(io.StringIO('[{"experiment":"e00_resnet18_3","macro_auc":0.590658743663053,"seed_sd":0.006534070160005,"bce":0.736453036467234,"seed42_auc":0.583330103534089,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"resnet18"},{"experiment":"e01_resnet18_5","macro_auc":0.606793164855707,"seed_sd":0.006085805010562,"bce":0.734935601552328,"seed42_auc":0.598210860201276,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"resnet18"},{"experiment":"e02a_dinov2_5","macro_auc":0.562761808749354,"seed_sd":0.017212452142254,"bce":0.727786421775818,"seed42_auc":0.577999668583191,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"dinov2"},{"experiment":"e02b_swin_5","macro_auc":0.619576657742699,"seed_sd":0.00321182713733,"bce":0.730335474014282,"seed42_auc":0.620172485120765,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"swin_t"},{"experiment":"e02c_convnext_5","macro_auc":0.59745041717041,"seed_sd":0.004088935880441,"bce":0.747565845648448,"seed42_auc":0.601623044900119,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"convnext_tiny"},{"experiment":"e02d_vit_5","macro_auc":0.593017946473149,"seed_sd":0.002950444238571,"bce":0.756685833136241,"seed42_auc":0.596596462155567,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"vit_b16"},{"experiment":"e03_attention","macro_auc":0.583633426406227,"seed_sd":0.03028233941894,"bce":0.662931183973948,"seed42_auc":0.590462878068234,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"swin_t"},{"experiment":"e04_position","macro_auc":0.575507370049808,"seed_sd":0.029291001232644,"bce":0.659927546977997,"seed42_auc":0.573035188534367,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"swin_t"},{"experiment":"e05_plane","macro_auc":0.575333904620255,"seed_sd":0.029755934497676,"bce":0.659805337587992,"seed42_auc":0.572689291413969,"gold_studies":58,"training_studies":58,"head_epochs":5,"backbone":"swin_t"},{"experiment":"e06_gold_control","macro_auc":0.585971944298197,"seed_sd":0.000882163475819,"bce":0.753049870332082,"seed42_auc":0.585539176849258,"gold_studies":58,"training_studies":58,"head_epochs":20,"backbone":"swin_t"},{"experiment":"e07_weak_labels","macro_auc":0.74194110470544,"seed_sd":0.004700688741065,"bce":0.583596845467885,"seed42_auc":0.735405849771202,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"e08_triplet","macro_auc":0.742925842493323,"seed_sd":0.001533901624984,"bce":0.576925853888194,"seed42_auc":0.740803040207208,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"e09_protocols","macro_auc":0.74411318037676,"seed_sd":0.001656732964649,"bce":0.596605181694031,"seed42_auc":0.744379169025354,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"e10_class_mil","macro_auc":0.713055260390097,"seed_sd":0.007476567009216,"bce":0.641398966312408,"seed42_auc":0.707472331228277,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"e11_finetune","macro_auc":0.713888527044199,"seed_sd":0.008375371973454,"bce":0.64122062921524,"seed42_auc":0.705724870429072,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"e12_ensemble","macro_auc":0.71974716605812,"seed_sd":0.007657737065486,"bce":0.625238716602326,"seed42_auc":0.713671829830429,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"r01_e09_s13_blend","macro_auc":0.771431855090667,"seed_sd":0.008915933806587,"bce":0.554393212000529,"seed42_auc":0.770804871736348,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s00_triplet","macro_auc":0.719499682252332,"seed_sd":0.008461178534246,"bce":0.664882481098175,"seed42_auc":0.728567322005846,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s01_slice3","macro_auc":0.71656296026774,"seed_sd":0.003434404997771,"bce":0.635896146297455,"seed42_auc":0.715464807554169,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s02_slice5","macro_auc":0.732540849043235,"seed_sd":0.012941281078386,"bce":0.614175796508789,"seed42_auc":0.740287833034452,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s03_slice10","macro_auc":0.724004706056847,"seed_sd":0.003790787943113,"bce":0.640817224979401,"seed42_auc":0.718663225471713,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s04_sequence","macro_auc":0.726607254777259,"seed_sd":0.009375761620229,"bce":0.623971819877624,"seed42_auc":0.735609354060918,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s05_position","macro_auc":0.713060729234292,"seed_sd":0.007051812198245,"bce":0.636056363582611,"seed42_auc":0.703189456655838,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s06_rope_index","macro_auc":0.721530303596713,"seed_sd":0.008818238578632,"bce":0.628619114557902,"seed42_auc":0.711719734921566,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s07_rope_mm","macro_auc":0.711794487175512,"seed_sd":0.017936358149139,"bce":0.639865795771281,"seed42_auc":0.688085750470196,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s08_plane","macro_auc":0.725969411146229,"seed_sd":0.014789936431291,"bce":0.616680463155111,"seed42_auc":0.70862727935855,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s09_mlp","macro_auc":0.724313989352095,"seed_sd":0.013194825535717,"bce":0.642311056454976,"seed42_auc":0.705656942656698,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s10_gate","macro_auc":0.739986229259686,"seed_sd":0.008870260193367,"bce":0.615018546581268,"seed42_auc":0.733655738137834,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s11_cross","macro_auc":0.736386419773609,"seed_sd":0.017465663239548,"bce":0.618532677491506,"seed42_auc":0.748004054111722,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s12_finetune","macro_auc":0.718887526942464,"seed_sd":0.010686567063529,"bce":0.648535231749217,"seed42_auc":0.724550567281684,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"},{"experiment":"s13_ensemble","macro_auc":0.748420070368692,"seed_sd":0.012975479398928,"bce":0.585482438405355,"seed42_auc":0.755714164774591,"gold_studies":58,"training_studies":4350,"head_epochs":20,"backbone":"swin_t"}]'))
category = pd.read_json(io.StringIO('[{"experiment":"r01_e09_s13_blend","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":26,"fp":8,"fn":8,"tp":16,"recall":0.666666666666667,"specificity":0.764705882352941,"precision":0.666666666666667,"f1":0.666666666666667,"auc":0.753676470588235,"average_precision":0.688977291314844},{"experiment":"r01_e09_s13_blend","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":40,"fp":9,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.816326530612245,"precision":0.181818181818182,"f1":0.2,"auc":0.619047619047619,"average_precision":0.381731394086417},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":19,"fp":13,"fn":5,"tp":21,"recall":0.807692307692308,"specificity":0.59375,"precision":0.61764705882353,"f1":0.7,"auc":0.788461538461538,"average_precision":0.792691083802584},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":28,"fp":7,"fn":15,"tp":8,"recall":0.347826086956522,"specificity":0.8,"precision":0.533333333333333,"f1":0.421052631578947,"auc":0.648447204968944,"average_precision":0.559048156186731},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":29,"fp":14,"fn":1,"tp":14,"recall":0.933333333333333,"specificity":0.674418604651163,"precision":0.5,"f1":0.651162790697674,"auc":0.916279069767442,"average_precision":0.814623644699047},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":30,"fp":17,"fn":2,"tp":9,"recall":0.818181818181818,"specificity":0.638297872340426,"precision":0.346153846153846,"f1":0.486486486486486,"auc":0.7678916827853,"average_precision":0.40148504751953},{"experiment":"r01_e09_s13_blend","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":25,"fp":12,"fn":5,"tp":16,"recall":0.761904761904762,"specificity":0.675675675675676,"precision":0.571428571428571,"f1":0.653061224489796,"auc":0.791505791505792,"average_precision":0.736538874215567},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":0,"tp":35,"recall":1.0,"specificity":0.478260869565217,"precision":0.74468085106383,"f1":0.853658536585366,"auc":0.88695652173913,"average_precision":0.926549825690795},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":15,"fp":16,"fn":5,"tp":22,"recall":0.814814814814815,"specificity":0.483870967741936,"precision":0.578947368421053,"f1":0.676923076923077,"auc":0.683393070489845,"average_precision":0.679735403710411},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":37,"fp":9,"fn":1,"tp":11,"recall":0.916666666666667,"specificity":0.804347826086956,"precision":0.55,"f1":0.6875,"auc":0.884057971014493,"average_precision":0.578166971916972},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":27,"fp":12,"fn":5,"tp":14,"recall":0.736842105263158,"specificity":0.692307692307692,"precision":0.538461538461538,"f1":0.622222222222222,"auc":0.754385964912281,"average_precision":0.655527510102618},{"experiment":"r01_e09_s13_blend","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":37,"fp":3,"fn":13,"tp":5,"recall":0.277777777777778,"specificity":0.925,"precision":0.625,"f1":0.384615384615385,"auc":0.755555555555556,"average_precision":0.605249771307993},{"experiment":"e03_attention","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":23,"fp":11,"fn":14,"tp":10,"recall":0.416666666666667,"specificity":0.676470588235294,"precision":0.476190476190476,"f1":0.444444444444444,"auc":0.579656862745098,"average_precision":0.469253630051754},{"experiment":"e03_attention","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":41,"fp":8,"fn":9,"tp":0,"recall":0.0,"specificity":0.836734693877551,"precision":0.0,"f1":0.0,"auc":0.433106575963719,"average_precision":0.146246979407382},{"experiment":"e03_attention","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":8,"tp":18,"recall":0.692307692307692,"specificity":0.5625,"precision":0.5625,"f1":0.620689655172414,"auc":0.622596153846154,"average_precision":0.537283729929497},{"experiment":"e03_attention","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":21,"fp":14,"fn":6,"tp":17,"recall":0.739130434782609,"specificity":0.6,"precision":0.548387096774194,"f1":0.62962962962963,"auc":0.663354037267081,"average_precision":0.551015282854368},{"experiment":"e03_attention","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":29,"fp":14,"fn":5,"tp":10,"recall":0.666666666666667,"specificity":0.674418604651163,"precision":0.416666666666667,"f1":0.512820512820513,"auc":0.731782945736434,"average_precision":0.657258529557024},{"experiment":"e03_attention","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":36,"fp":11,"fn":9,"tp":2,"recall":0.181818181818182,"specificity":0.765957446808511,"precision":0.153846153846154,"f1":0.166666666666667,"auc":0.642166344294004,"average_precision":0.304744837243842},{"experiment":"e03_attention","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":19,"fp":18,"fn":9,"tp":12,"recall":0.571428571428571,"specificity":0.513513513513514,"precision":0.4,"f1":0.470588235294118,"auc":0.571428571428571,"average_precision":0.506508367772644},{"experiment":"e03_attention","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":14,"fp":9,"fn":7,"tp":28,"recall":0.8,"specificity":0.608695652173913,"precision":0.756756756756757,"f1":0.777777777777778,"auc":0.732919254658385,"average_precision":0.806848790743294},{"experiment":"e03_attention","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":14,"fp":17,"fn":8,"tp":19,"recall":0.703703703703704,"specificity":0.451612903225806,"precision":0.527777777777778,"f1":0.603174603174603,"auc":0.637992831541219,"average_precision":0.56192152405233},{"experiment":"e03_attention","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":23,"fp":23,"fn":9,"tp":3,"recall":0.25,"specificity":0.5,"precision":0.115384615384615,"f1":0.157894736842105,"auc":0.403985507246377,"average_precision":0.255658739077482},{"experiment":"e03_attention","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":16,"fp":23,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.41025641025641,"precision":0.361111111111111,"f1":0.472727272727273,"auc":0.574898785425101,"average_precision":0.434932244943154},{"experiment":"e03_attention","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":20,"fp":20,"fn":8,"tp":10,"recall":0.555555555555556,"specificity":0.5,"precision":0.333333333333333,"f1":0.416666666666667,"auc":0.491666666666667,"average_precision":0.302793267154116},{"experiment":"e04_position","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":16,"fp":18,"fn":6,"tp":18,"recall":0.75,"specificity":0.470588235294118,"precision":0.5,"f1":0.6,"auc":0.579656862745098,"average_precision":0.459197414901621},{"experiment":"e04_position","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":38,"fp":11,"fn":9,"tp":0,"recall":0.0,"specificity":0.775510204081633,"precision":0.0,"f1":0.0,"auc":0.412698412698413,"average_precision":0.139208942703095},{"experiment":"e04_position","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":19,"fp":13,"fn":8,"tp":18,"recall":0.692307692307692,"specificity":0.59375,"precision":0.580645161290323,"f1":0.631578947368421,"auc":0.622596153846154,"average_precision":0.554963728015803},{"experiment":"e04_position","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":19,"fp":16,"fn":9,"tp":14,"recall":0.608695652173913,"specificity":0.542857142857143,"precision":0.466666666666667,"f1":0.528301886792453,"auc":0.636024844720497,"average_precision":0.527767122288503},{"experiment":"e04_position","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":31,"fp":12,"fn":6,"tp":9,"recall":0.6,"specificity":0.72093023255814,"precision":0.428571428571428,"f1":0.5,"auc":0.713178294573644,"average_precision":0.628503589901536},{"experiment":"e04_position","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":39,"fp":8,"fn":9,"tp":2,"recall":0.181818181818182,"specificity":0.829787234042553,"precision":0.2,"f1":0.19047619047619,"auc":0.661508704061896,"average_precision":0.359844896937323},{"experiment":"e04_position","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":22,"fp":15,"fn":9,"tp":12,"recall":0.571428571428571,"specificity":0.594594594594595,"precision":0.444444444444444,"f1":0.5,"auc":0.564993564993565,"average_precision":0.500995886031736},{"experiment":"e04_position","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":16,"fp":7,"fn":13,"tp":22,"recall":0.628571428571428,"specificity":0.695652173913044,"precision":0.758620689655172,"f1":0.6875,"auc":0.721739130434783,"average_precision":0.763972880674757},{"experiment":"e04_position","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":18,"fp":13,"fn":10,"tp":17,"recall":0.62962962962963,"specificity":0.580645161290323,"precision":0.566666666666667,"f1":0.596491228070175,"auc":0.60573476702509,"average_precision":0.541985386418487},{"experiment":"e04_position","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":30,"fp":16,"fn":10,"tp":2,"recall":0.166666666666667,"specificity":0.652173913043478,"precision":0.111111111111111,"f1":0.133333333333333,"auc":0.391304347826087,"average_precision":0.253168484451918},{"experiment":"e04_position","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":31,"fp":8,"fn":16,"tp":3,"recall":0.157894736842105,"specificity":0.794871794871795,"precision":0.272727272727273,"f1":0.2,"auc":0.512820512820513,"average_precision":0.347502382464083},{"experiment":"e04_position","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":26,"fp":14,"fn":14,"tp":4,"recall":0.222222222222222,"specificity":0.65,"precision":0.222222222222222,"f1":0.222222222222222,"auc":0.454166666666667,"average_precision":0.282262671178487},{"experiment":"e05_plane","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":16,"fp":18,"fn":6,"tp":18,"recall":0.75,"specificity":0.470588235294118,"precision":0.5,"f1":0.6,"auc":0.580882352941176,"average_precision":0.460293906129692},{"experiment":"e05_plane","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":38,"fp":11,"fn":9,"tp":0,"recall":0.0,"specificity":0.775510204081633,"precision":0.0,"f1":0.0,"auc":0.412698412698413,"average_precision":0.139208942703095},{"experiment":"e05_plane","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":19,"fp":13,"fn":8,"tp":18,"recall":0.692307692307692,"specificity":0.59375,"precision":0.580645161290323,"f1":0.631578947368421,"auc":0.622596153846154,"average_precision":0.554963728015803},{"experiment":"e05_plane","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":19,"fp":16,"fn":9,"tp":14,"recall":0.608695652173913,"specificity":0.542857142857143,"precision":0.466666666666667,"f1":0.528301886792453,"auc":0.634782608695652,"average_precision":0.527209708687611},{"experiment":"e05_plane","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":31,"fp":12,"fn":6,"tp":9,"recall":0.6,"specificity":0.72093023255814,"precision":0.428571428571428,"f1":0.5,"auc":0.711627906976744,"average_precision":0.624261165659112},{"experiment":"e05_plane","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":39,"fp":8,"fn":9,"tp":2,"recall":0.181818181818182,"specificity":0.829787234042553,"precision":0.2,"f1":0.19047619047619,"auc":0.661508704061896,"average_precision":0.359844896937323},{"experiment":"e05_plane","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":22,"fp":15,"fn":9,"tp":12,"recall":0.571428571428571,"specificity":0.594594594594595,"precision":0.444444444444444,"f1":0.5,"auc":0.564993564993565,"average_precision":0.500995886031736},{"experiment":"e05_plane","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":16,"fp":7,"fn":13,"tp":22,"recall":0.628571428571428,"specificity":0.695652173913044,"precision":0.758620689655172,"f1":0.6875,"auc":0.721739130434783,"average_precision":0.763972880674757},{"experiment":"e05_plane","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":18,"fp":13,"fn":10,"tp":17,"recall":0.62962962962963,"specificity":0.580645161290323,"precision":0.566666666666667,"f1":0.596491228070175,"auc":0.604540023894863,"average_precision":0.541255592585245},{"experiment":"e05_plane","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":30,"fp":16,"fn":10,"tp":2,"recall":0.166666666666667,"specificity":0.652173913043478,"precision":0.111111111111111,"f1":0.133333333333333,"auc":0.391304347826087,"average_precision":0.253168484451918},{"experiment":"e05_plane","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":31,"fp":8,"fn":16,"tp":3,"recall":0.157894736842105,"specificity":0.794871794871795,"precision":0.272727272727273,"f1":0.2,"auc":0.512820512820513,"average_precision":0.347502382464083},{"experiment":"e05_plane","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":26,"fp":14,"fn":14,"tp":4,"recall":0.222222222222222,"specificity":0.65,"precision":0.222222222222222,"f1":0.222222222222222,"auc":0.452777777777778,"average_precision":0.281661420577236},{"experiment":"s00_triplet","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":21,"fp":13,"fn":10,"tp":14,"recall":0.583333333333333,"specificity":0.61764705882353,"precision":0.518518518518518,"f1":0.549019607843137,"auc":0.707107843137255,"average_precision":0.611904865306221},{"experiment":"s00_triplet","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":42,"fp":7,"fn":6,"tp":3,"recall":0.333333333333333,"specificity":0.857142857142857,"precision":0.3,"f1":0.31578947368421,"auc":0.571428571428571,"average_precision":0.249913575416671},{"experiment":"s00_triplet","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":21,"fp":11,"fn":4,"tp":22,"recall":0.846153846153846,"specificity":0.65625,"precision":0.666666666666667,"f1":0.745762711864407,"auc":0.766826923076923,"average_precision":0.63632468383925},{"experiment":"s00_triplet","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":27,"fp":8,"fn":14,"tp":9,"recall":0.391304347826087,"specificity":0.771428571428572,"precision":0.529411764705882,"f1":0.45,"auc":0.537888198757764,"average_precision":0.460639870972622},{"experiment":"s00_triplet","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":27,"fp":16,"fn":2,"tp":13,"recall":0.866666666666667,"specificity":0.627906976744186,"precision":0.448275862068966,"f1":0.590909090909091,"auc":0.852713178294574,"average_precision":0.661695536395426},{"experiment":"s00_triplet","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":30,"fp":17,"fn":3,"tp":8,"recall":0.727272727272727,"specificity":0.638297872340426,"precision":0.32,"f1":0.444444444444444,"auc":0.709864603481625,"average_precision":0.38286583604967},{"experiment":"s00_triplet","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":22,"fp":15,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.594594594594595,"precision":0.53125,"f1":0.641509433962264,"auc":0.799227799227799,"average_precision":0.707139701200654},{"experiment":"s00_triplet","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":10,"fp":13,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.434782608695652,"precision":0.704545454545455,"f1":0.784810126582278,"auc":0.793788819875776,"average_precision":0.870047900579021},{"experiment":"s00_triplet","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":17,"fp":14,"fn":9,"tp":18,"recall":0.666666666666667,"specificity":0.548387096774194,"precision":0.5625,"f1":0.610169491525424,"auc":0.741935483870968,"average_precision":0.716196407633798},{"experiment":"s00_triplet","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":36,"fp":10,"fn":4,"tp":8,"recall":0.666666666666667,"specificity":0.782608695652174,"precision":0.444444444444444,"f1":0.533333333333333,"auc":0.778985507246377,"average_precision":0.469198091256915},{"experiment":"s00_triplet","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":24,"fp":15,"fn":2,"tp":17,"recall":0.894736842105263,"specificity":0.615384615384615,"precision":0.53125,"f1":0.666666666666667,"auc":0.771929824561404,"average_precision":0.588512379046833},{"experiment":"s00_triplet","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":37,"fp":3,"fn":16,"tp":2,"recall":0.111111111111111,"specificity":0.925,"precision":0.4,"f1":0.173913043478261,"auc":0.711111111111111,"average_precision":0.513220788455851},{"experiment":"s01_slice3","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":24,"fp":10,"fn":8,"tp":16,"recall":0.666666666666667,"specificity":0.705882352941176,"precision":0.615384615384615,"f1":0.64,"auc":0.72671568627451,"average_precision":0.642008057280439},{"experiment":"s01_slice3","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":41,"fp":8,"fn":8,"tp":1,"recall":0.111111111111111,"specificity":0.836734693877551,"precision":0.111111111111111,"f1":0.111111111111111,"auc":0.585034013605442,"average_precision":0.24913020260538},{"experiment":"s01_slice3","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":16,"fp":16,"fn":3,"tp":23,"recall":0.884615384615385,"specificity":0.5,"precision":0.58974358974359,"f1":0.707692307692308,"auc":0.769230769230769,"average_precision":0.739263861177579},{"experiment":"s01_slice3","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":24,"fp":11,"fn":14,"tp":9,"recall":0.391304347826087,"specificity":0.685714285714286,"precision":0.45,"f1":0.418604651162791,"auc":0.508074534161491,"average_precision":0.43831281824272},{"experiment":"s01_slice3","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":2,"tp":13,"recall":0.866666666666667,"specificity":0.697674418604651,"precision":0.5,"f1":0.634146341463415,"auc":0.897674418604651,"average_precision":0.794344784482039},{"experiment":"s01_slice3","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":29,"fp":18,"fn":3,"tp":8,"recall":0.727272727272727,"specificity":0.617021276595745,"precision":0.307692307692308,"f1":0.432432432432432,"auc":0.690522243713733,"average_precision":0.413257779724769},{"experiment":"s01_slice3","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":25,"fp":12,"fn":7,"tp":14,"recall":0.666666666666667,"specificity":0.675675675675676,"precision":0.538461538461538,"f1":0.595744680851064,"auc":0.756756756756757,"average_precision":0.704653312722407},{"experiment":"s01_slice3","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":10,"fp":13,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.434782608695652,"precision":0.704545454545455,"f1":0.784810126582278,"auc":0.783850931677019,"average_precision":0.832598008828951},{"experiment":"s01_slice3","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":19,"fp":12,"fn":11,"tp":16,"recall":0.592592592592592,"specificity":0.612903225806452,"precision":0.571428571428571,"f1":0.581818181818182,"auc":0.62962962962963,"average_precision":0.582761367742518},{"experiment":"s01_slice3","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":37,"fp":9,"fn":3,"tp":9,"recall":0.75,"specificity":0.804347826086956,"precision":0.5,"f1":0.6,"auc":0.840579710144928,"average_precision":0.659895354827309},{"experiment":"s01_slice3","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":23,"fp":16,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.58974358974359,"precision":0.448275862068966,"f1":0.541666666666667,"auc":0.704453441295546,"average_precision":0.564049693933196},{"experiment":"s01_slice3","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":37,"fp":3,"fn":12,"tp":6,"recall":0.333333333333333,"specificity":0.925,"precision":0.666666666666667,"f1":0.444444444444444,"auc":0.693055555555556,"average_precision":0.55303449550751},{"experiment":"s02_slice5","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":25,"fp":9,"fn":3,"tp":21,"recall":0.875,"specificity":0.735294117647059,"precision":0.7,"f1":0.777777777777778,"auc":0.801470588235294,"average_precision":0.667545881152186},{"experiment":"s02_slice5","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":43,"fp":6,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.877551020408163,"precision":0.25,"f1":0.235294117647059,"auc":0.643990929705215,"average_precision":0.307539132539132},{"experiment":"s02_slice5","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":17,"fp":15,"fn":5,"tp":21,"recall":0.807692307692308,"specificity":0.53125,"precision":0.583333333333333,"f1":0.67741935483871,"auc":0.769230769230769,"average_precision":0.710987761834983},{"experiment":"s02_slice5","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":27,"fp":8,"fn":15,"tp":8,"recall":0.347826086956522,"specificity":0.771428571428572,"precision":0.5,"f1":0.41025641025641,"auc":0.618633540372671,"average_precision":0.5028407849517},{"experiment":"s02_slice5","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":3,"tp":12,"recall":0.8,"specificity":0.697674418604651,"precision":0.48,"f1":0.6,"auc":0.882170542635659,"average_precision":0.781356134511907},{"experiment":"s02_slice5","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":30,"fp":17,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.638297872340426,"precision":0.260869565217391,"f1":0.352941176470588,"auc":0.754352030947776,"average_precision":0.501777902812386},{"experiment":"s02_slice5","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":26,"fp":11,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.702702702702703,"precision":0.607142857142857,"f1":0.693877551020408,"auc":0.806949806949807,"average_precision":0.699137063960728},{"experiment":"s02_slice5","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":10,"fp":13,"fn":3,"tp":32,"recall":0.914285714285714,"specificity":0.434782608695652,"precision":0.711111111111111,"f1":0.8,"auc":0.818633540372671,"average_precision":0.879182323410146},{"experiment":"s02_slice5","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":18,"fp":13,"fn":11,"tp":16,"recall":0.592592592592592,"specificity":0.580645161290323,"precision":0.551724137931034,"f1":0.571428571428571,"auc":0.584229390681004,"average_precision":0.556224273416666},{"experiment":"s02_slice5","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":39,"fp":7,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.847826086956522,"precision":0.5,"f1":0.538461538461538,"auc":0.838768115942029,"average_precision":0.640362954946288},{"experiment":"s02_slice5","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":25,"fp":14,"fn":5,"tp":14,"recall":0.736842105263158,"specificity":0.641025641025641,"precision":0.5,"f1":0.595744680851064,"auc":0.770580296896086,"average_precision":0.618920425596222},{"experiment":"s02_slice5","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":35,"fp":5,"fn":12,"tp":6,"recall":0.333333333333333,"specificity":0.875,"precision":0.545454545454545,"f1":0.413793103448276,"auc":0.594444444444444,"average_precision":0.52530097601735},{"experiment":"s03_slice10","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":25,"fp":9,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.735294117647059,"precision":0.653846153846154,"f1":0.68,"auc":0.792892156862745,"average_precision":0.682556225197741},{"experiment":"s03_slice10","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":41,"fp":8,"fn":8,"tp":1,"recall":0.111111111111111,"specificity":0.836734693877551,"precision":0.111111111111111,"f1":0.111111111111111,"auc":0.580498866213152,"average_precision":0.191816589490121},{"experiment":"s03_slice10","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":14,"fp":18,"fn":4,"tp":22,"recall":0.846153846153846,"specificity":0.4375,"precision":0.55,"f1":0.666666666666667,"auc":0.727163461538462,"average_precision":0.720393960295867},{"experiment":"s03_slice10","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":18,"fp":17,"fn":14,"tp":9,"recall":0.391304347826087,"specificity":0.514285714285714,"precision":0.346153846153846,"f1":0.36734693877551,"auc":0.526708074534161,"average_precision":0.467894128158886},{"experiment":"s03_slice10","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":33,"fp":10,"fn":5,"tp":10,"recall":0.666666666666667,"specificity":0.767441860465116,"precision":0.5,"f1":0.571428571428571,"auc":0.84031007751938,"average_precision":0.701660370195544},{"experiment":"s03_slice10","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":30,"fp":17,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.638297872340426,"precision":0.291666666666667,"f1":0.4,"auc":0.704061895551257,"average_precision":0.459505406873828},{"experiment":"s03_slice10","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":26,"fp":11,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.702702702702703,"precision":0.607142857142857,"f1":0.693877551020408,"auc":0.808236808236808,"average_precision":0.750078261037645},{"experiment":"s03_slice10","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.478260869565217,"precision":0.72093023255814,"f1":0.794871794871795,"auc":0.857142857142857,"average_precision":0.909525274839537},{"experiment":"s03_slice10","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":18,"fp":13,"fn":11,"tp":16,"recall":0.592592592592592,"specificity":0.580645161290323,"precision":0.551724137931034,"f1":0.571428571428571,"auc":0.597371565113501,"average_precision":0.588426859501577},{"experiment":"s03_slice10","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":38,"fp":8,"fn":4,"tp":8,"recall":0.666666666666667,"specificity":0.826086956521739,"precision":0.5,"f1":0.571428571428571,"auc":0.829710144927536,"average_precision":0.688859649122807},{"experiment":"s03_slice10","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":29,"fp":10,"fn":7,"tp":12,"recall":0.631578947368421,"specificity":0.743589743589744,"precision":0.545454545454545,"f1":0.585365853658536,"auc":0.757085020242915,"average_precision":0.536277932081511},{"experiment":"s03_slice10","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":36,"fp":4,"fn":14,"tp":4,"recall":0.222222222222222,"specificity":0.9,"precision":0.5,"f1":0.307692307692308,"auc":0.602777777777778,"average_precision":0.493735827859025},{"experiment":"s04_sequence","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":24,"fp":10,"fn":6,"tp":18,"recall":0.75,"specificity":0.705882352941176,"precision":0.642857142857143,"f1":0.692307692307692,"auc":0.763480392156863,"average_precision":0.68976876717837},{"experiment":"s04_sequence","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":43,"fp":6,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.877551020408163,"precision":0.25,"f1":0.235294117647059,"auc":0.63718820861678,"average_precision":0.24133831451466},{"experiment":"s04_sequence","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":21,"fp":11,"fn":9,"tp":17,"recall":0.653846153846154,"specificity":0.65625,"precision":0.607142857142857,"f1":0.62962962962963,"auc":0.723557692307692,"average_precision":0.713621711417708},{"experiment":"s04_sequence","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":22,"fp":13,"fn":11,"tp":12,"recall":0.521739130434783,"specificity":0.628571428571428,"precision":0.48,"f1":0.5,"auc":0.64472049689441,"average_precision":0.533726781189919},{"experiment":"s04_sequence","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":3,"tp":12,"recall":0.8,"specificity":0.697674418604651,"precision":0.48,"f1":0.6,"auc":0.813953488372093,"average_precision":0.611217580685666},{"experiment":"s04_sequence","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":31,"fp":16,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.659574468085106,"precision":0.272727272727273,"f1":0.363636363636364,"auc":0.702127659574468,"average_precision":0.438418884541879},{"experiment":"s04_sequence","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":27,"fp":10,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.72972972972973,"precision":0.62962962962963,"f1":0.708333333333333,"auc":0.821106821106821,"average_precision":0.692806955826064},{"experiment":"s04_sequence","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":0,"tp":35,"recall":1.0,"specificity":0.478260869565217,"precision":0.74468085106383,"f1":0.853658536585366,"auc":0.808695652173913,"average_precision":0.862261522033878},{"experiment":"s04_sequence","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":13,"fp":18,"fn":8,"tp":19,"recall":0.703703703703704,"specificity":0.419354838709677,"precision":0.513513513513514,"f1":0.59375,"auc":0.659498207885305,"average_precision":0.692864215064888},{"experiment":"s04_sequence","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":37,"fp":9,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.804347826086956,"precision":0.4375,"f1":0.5,"auc":0.818840579710145,"average_precision":0.491248403085741},{"experiment":"s04_sequence","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":24,"fp":15,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.615384615384615,"precision":0.464285714285714,"f1":0.553191489361702,"auc":0.734143049932524,"average_precision":0.603776701380657},{"experiment":"s04_sequence","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":35,"fp":5,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.875,"precision":0.583333333333333,"f1":0.466666666666667,"auc":0.7,"average_precision":0.613047055059634},{"experiment":"s05_position","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":21,"fp":13,"fn":6,"tp":18,"recall":0.75,"specificity":0.61764705882353,"precision":0.580645161290323,"f1":0.654545454545454,"auc":0.720588235294118,"average_precision":0.567348483599258},{"experiment":"s05_position","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":45,"fp":4,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.918367346938776,"precision":0.333333333333333,"f1":0.266666666666667,"auc":0.562358276643991,"average_precision":0.236778605761844},{"experiment":"s05_position","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":19,"fp":13,"fn":8,"tp":18,"recall":0.692307692307692,"specificity":0.59375,"precision":0.580645161290323,"f1":0.631578947368421,"auc":0.681490384615385,"average_precision":0.57226539033859},{"experiment":"s05_position","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":23,"fp":12,"fn":13,"tp":10,"recall":0.434782608695652,"specificity":0.657142857142857,"precision":0.454545454545454,"f1":0.444444444444444,"auc":0.524223602484472,"average_precision":0.470976264447609},{"experiment":"s05_position","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":28,"fp":15,"fn":3,"tp":12,"recall":0.8,"specificity":0.651162790697674,"precision":0.444444444444444,"f1":0.571428571428571,"auc":0.843410852713178,"average_precision":0.607592320533497},{"experiment":"s05_position","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":34,"fp":13,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.723404255319149,"precision":0.35,"f1":0.451612903225806,"auc":0.748549323017408,"average_precision":0.3590882021983},{"experiment":"s05_position","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":25,"fp":12,"fn":3,"tp":18,"recall":0.857142857142857,"specificity":0.675675675675676,"precision":0.6,"f1":0.705882352941176,"auc":0.88030888030888,"average_precision":0.8050855314619},{"experiment":"s05_position","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":8,"fp":15,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.347826086956522,"precision":0.673913043478261,"f1":0.765432098765432,"auc":0.727950310559006,"average_precision":0.787327655053887},{"experiment":"s05_position","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":14,"fp":17,"fn":10,"tp":17,"recall":0.62962962962963,"specificity":0.451612903225806,"precision":0.5,"f1":0.557377049180328,"auc":0.589008363201912,"average_precision":0.600940996875714},{"experiment":"s05_position","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":35,"fp":11,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.760869565217391,"precision":0.388888888888889,"f1":0.466666666666667,"auc":0.769927536231884,"average_precision":0.475724250523012},{"experiment":"s05_position","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":23,"fp":16,"fn":4,"tp":15,"recall":0.789473684210526,"specificity":0.58974358974359,"precision":0.483870967741936,"f1":0.6,"auc":0.755735492577598,"average_precision":0.587425327539651},{"experiment":"s05_position","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":35,"fp":5,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.875,"precision":0.583333333333333,"f1":0.466666666666667,"auc":0.634722222222222,"average_precision":0.554862619695896},{"experiment":"s06_rope_index","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":27,"fp":7,"fn":9,"tp":15,"recall":0.625,"specificity":0.794117647058824,"precision":0.681818181818182,"f1":0.652173913043478,"auc":0.70343137254902,"average_precision":0.627783610352145},{"experiment":"s06_rope_index","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":40,"fp":9,"fn":8,"tp":1,"recall":0.111111111111111,"specificity":0.816326530612245,"precision":0.1,"f1":0.105263157894737,"auc":0.557823129251701,"average_precision":0.19072448153805},{"experiment":"s06_rope_index","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":22,"fp":10,"fn":10,"tp":16,"recall":0.615384615384615,"specificity":0.6875,"precision":0.615384615384615,"f1":0.615384615384615,"auc":0.717548076923077,"average_precision":0.620576324910619},{"experiment":"s06_rope_index","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":22,"fp":13,"fn":12,"tp":11,"recall":0.478260869565217,"specificity":0.628571428571428,"precision":0.458333333333333,"f1":0.468085106382979,"auc":0.583850931677019,"average_precision":0.495727071479379},{"experiment":"s06_rope_index","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":28,"fp":15,"fn":3,"tp":12,"recall":0.8,"specificity":0.651162790697674,"precision":0.444444444444444,"f1":0.571428571428571,"auc":0.869767441860465,"average_precision":0.803439284729607},{"experiment":"s06_rope_index","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":34,"fp":13,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.723404255319149,"precision":0.35,"f1":0.451612903225806,"auc":0.752417794970986,"average_precision":0.473505008063832},{"experiment":"s06_rope_index","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":26,"fp":11,"fn":5,"tp":16,"recall":0.761904761904762,"specificity":0.702702702702703,"precision":0.592592592592592,"f1":0.666666666666667,"auc":0.81981981981982,"average_precision":0.737986893899971},{"experiment":"s06_rope_index","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.478260869565217,"precision":0.72093023255814,"f1":0.794871794871795,"auc":0.82360248447205,"average_precision":0.845491614278096},{"experiment":"s06_rope_index","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":16,"fp":15,"fn":10,"tp":17,"recall":0.62962962962963,"specificity":0.516129032258064,"precision":0.53125,"f1":0.576271186440678,"auc":0.61768219832736,"average_precision":0.595492696753346},{"experiment":"s06_rope_index","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":36,"fp":10,"fn":6,"tp":6,"recall":0.5,"specificity":0.782608695652174,"precision":0.375,"f1":0.428571428571428,"auc":0.742753623188406,"average_precision":0.535595404342967},{"experiment":"s06_rope_index","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":23,"fp":16,"fn":8,"tp":11,"recall":0.578947368421053,"specificity":0.58974358974359,"precision":0.407407407407407,"f1":0.478260869565217,"auc":0.647773279352227,"average_precision":0.505312297897999},{"experiment":"s06_rope_index","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":36,"fp":4,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.9,"precision":0.636363636363636,"f1":0.482758620689655,"auc":0.704166666666667,"average_precision":0.606947893610256},{"experiment":"s07_rope_mm","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":24,"fp":10,"fn":10,"tp":14,"recall":0.583333333333333,"specificity":0.705882352941176,"precision":0.583333333333333,"f1":0.583333333333333,"auc":0.667892156862745,"average_precision":0.555127085442968},{"experiment":"s07_rope_mm","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":41,"fp":8,"fn":8,"tp":1,"recall":0.111111111111111,"specificity":0.836734693877551,"precision":0.111111111111111,"f1":0.111111111111111,"auc":0.541950113378685,"average_precision":0.21913401279365},{"experiment":"s07_rope_mm","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":9,"tp":17,"recall":0.653846153846154,"specificity":0.5625,"precision":0.548387096774194,"f1":0.596491228070175,"auc":0.662259615384615,"average_precision":0.601248315253275},{"experiment":"s07_rope_mm","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":20,"fp":15,"fn":13,"tp":10,"recall":0.434782608695652,"specificity":0.571428571428571,"precision":0.4,"f1":0.416666666666667,"auc":0.53167701863354,"average_precision":0.508983697182073},{"experiment":"s07_rope_mm","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":3,"tp":12,"recall":0.8,"specificity":0.697674418604651,"precision":0.48,"f1":0.6,"auc":0.827906976744186,"average_precision":0.69571238094439},{"experiment":"s07_rope_mm","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":34,"fp":13,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.723404255319149,"precision":0.31578947368421,"f1":0.4,"auc":0.752417794970986,"average_precision":0.460434456953646},{"experiment":"s07_rope_mm","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":27,"fp":10,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.72972972972973,"precision":0.62962962962963,"f1":0.708333333333333,"auc":0.815958815958816,"average_precision":0.787705673082014},{"experiment":"s07_rope_mm","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":8,"fp":15,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.347826086956522,"precision":0.673913043478261,"f1":0.765432098765432,"auc":0.77888198757764,"average_precision":0.851663979319974},{"experiment":"s07_rope_mm","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":12,"fp":19,"fn":9,"tp":18,"recall":0.666666666666667,"specificity":0.387096774193548,"precision":0.486486486486486,"f1":0.5625,"auc":0.568697729988053,"average_precision":0.571246137683051},{"experiment":"s07_rope_mm","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":34,"fp":12,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.739130434782609,"precision":0.368421052631579,"f1":0.451612903225806,"auc":0.753623188405797,"average_precision":0.494168941731282},{"experiment":"s07_rope_mm","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":19,"fp":20,"fn":5,"tp":14,"recall":0.736842105263158,"specificity":0.487179487179487,"precision":0.411764705882353,"f1":0.528301886792453,"auc":0.707152496626181,"average_precision":0.581240803578476},{"experiment":"s07_rope_mm","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":31,"fp":9,"fn":10,"tp":8,"recall":0.444444444444444,"specificity":0.775,"precision":0.470588235294118,"f1":0.457142857142857,"auc":0.648611111111111,"average_precision":0.550955594985059},{"experiment":"s08_plane","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":26,"fp":8,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.764705882352941,"precision":0.68,"f1":0.693877551020408,"auc":0.791666666666667,"average_precision":0.709630196156206},{"experiment":"s08_plane","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":43,"fp":6,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.877551020408163,"precision":0.25,"f1":0.235294117647059,"auc":0.551020408163265,"average_precision":0.278627699486471},{"experiment":"s08_plane","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":12,"tp":14,"recall":0.538461538461538,"specificity":0.5625,"precision":0.5,"f1":0.518518518518518,"auc":0.680288461538462,"average_precision":0.644956144106134},{"experiment":"s08_plane","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":23,"fp":12,"fn":16,"tp":7,"recall":0.304347826086956,"specificity":0.657142857142857,"precision":0.368421052631579,"f1":0.333333333333333,"auc":0.580124223602484,"average_precision":0.449398266090748},{"experiment":"s08_plane","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":32,"fp":11,"fn":3,"tp":12,"recall":0.8,"specificity":0.744186046511628,"precision":0.521739130434783,"f1":0.631578947368421,"auc":0.827906976744186,"average_precision":0.678948326019264},{"experiment":"s08_plane","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":30,"fp":17,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.638297872340426,"precision":0.260869565217391,"f1":0.352941176470588,"auc":0.62669245647969,"average_precision":0.278000086038364},{"experiment":"s08_plane","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":28,"fp":9,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.756756756756757,"precision":0.653846153846154,"f1":0.723404255319149,"auc":0.799227799227799,"average_precision":0.688024098050326},{"experiment":"s08_plane","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":7,"fp":16,"fn":2,"tp":33,"recall":0.942857142857143,"specificity":0.304347826086956,"precision":0.673469387755102,"f1":0.785714285714286,"auc":0.807453416149068,"average_precision":0.887016905983521},{"experiment":"s08_plane","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":20,"fp":11,"fn":6,"tp":21,"recall":0.777777777777778,"specificity":0.645161290322581,"precision":0.65625,"f1":0.711864406779661,"auc":0.671445639187575,"average_precision":0.603076823888068},{"experiment":"s08_plane","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":35,"fp":11,"fn":4,"tp":8,"recall":0.666666666666667,"specificity":0.760869565217391,"precision":0.421052631578947,"f1":0.516129032258064,"auc":0.833333333333333,"average_precision":0.655852528682225},{"experiment":"s08_plane","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":23,"fp":16,"fn":4,"tp":15,"recall":0.789473684210526,"specificity":0.58974358974359,"precision":0.483870967741936,"f1":0.6,"auc":0.678812415654521,"average_precision":0.490644179243218},{"experiment":"s08_plane","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":35,"fp":5,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.875,"precision":0.583333333333333,"f1":0.466666666666667,"auc":0.655555555555556,"average_precision":0.56515556643774},{"experiment":"s09_mlp","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":21,"fp":13,"fn":8,"tp":16,"recall":0.666666666666667,"specificity":0.61764705882353,"precision":0.551724137931034,"f1":0.60377358490566,"auc":0.674019607843137,"average_precision":0.58814210402379},{"experiment":"s09_mlp","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":39,"fp":10,"fn":9,"tp":0,"recall":0.0,"specificity":0.795918367346939,"precision":0.0,"f1":0.0,"auc":0.45124716553288,"average_precision":0.154877918905837},{"experiment":"s09_mlp","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":19,"fp":13,"fn":6,"tp":20,"recall":0.769230769230769,"specificity":0.59375,"precision":0.606060606060606,"f1":0.677966101694915,"auc":0.740384615384615,"average_precision":0.688833394525773},{"experiment":"s09_mlp","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":26,"fp":9,"fn":12,"tp":11,"recall":0.478260869565217,"specificity":0.742857142857143,"precision":0.55,"f1":0.511627906976744,"auc":0.663354037267081,"average_precision":0.58536516668732},{"experiment":"s09_mlp","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":0,"tp":15,"recall":1.0,"specificity":0.697674418604651,"precision":0.535714285714286,"f1":0.697674418604651,"auc":0.908527131782946,"average_precision":0.735575398686854},{"experiment":"s09_mlp","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":30,"fp":17,"fn":3,"tp":8,"recall":0.727272727272727,"specificity":0.638297872340426,"precision":0.32,"f1":0.444444444444444,"auc":0.742746615087041,"average_precision":0.577722863686357},{"experiment":"s09_mlp","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":19,"fp":18,"fn":6,"tp":15,"recall":0.714285714285714,"specificity":0.513513513513514,"precision":0.454545454545454,"f1":0.555555555555556,"auc":0.684684684684685,"average_precision":0.621084700909645},{"experiment":"s09_mlp","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":5,"fp":18,"fn":0,"tp":35,"recall":1.0,"specificity":0.217391304347826,"precision":0.660377358490566,"f1":0.795454545454545,"auc":0.746583850931677,"average_precision":0.829008550839986},{"experiment":"s09_mlp","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":12,"fp":19,"fn":3,"tp":24,"recall":0.888888888888889,"specificity":0.387096774193548,"precision":0.558139534883721,"f1":0.685714285714286,"auc":0.781362007168459,"average_precision":0.742614893790078},{"experiment":"s09_mlp","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":31,"fp":15,"fn":3,"tp":9,"recall":0.75,"specificity":0.673913043478261,"precision":0.375,"f1":0.5,"auc":0.742753623188406,"average_precision":0.395549921093399},{"experiment":"s09_mlp","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":17,"fp":22,"fn":4,"tp":15,"recall":0.789473684210526,"specificity":0.435897435897436,"precision":0.405405405405405,"f1":0.535714285714286,"auc":0.657219973009447,"average_precision":0.44794747794886},{"experiment":"s09_mlp","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":34,"fp":6,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.85,"precision":0.538461538461538,"f1":0.451612903225806,"auc":0.675,"average_precision":0.467491045954276},{"experiment":"s10_gate","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":24,"fp":10,"fn":10,"tp":14,"recall":0.583333333333333,"specificity":0.705882352941176,"precision":0.583333333333333,"f1":0.583333333333333,"auc":0.654411764705882,"average_precision":0.566974115875167},{"experiment":"s10_gate","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":41,"fp":8,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.836734693877551,"precision":0.2,"f1":0.210526315789474,"auc":0.571428571428571,"average_precision":0.291402892299716},{"experiment":"s10_gate","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":6,"tp":20,"recall":0.769230769230769,"specificity":0.5625,"precision":0.588235294117647,"f1":0.666666666666667,"auc":0.75,"average_precision":0.704744574207526},{"experiment":"s10_gate","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":31,"fp":4,"fn":16,"tp":7,"recall":0.304347826086956,"specificity":0.885714285714286,"precision":0.636363636363636,"f1":0.411764705882353,"auc":0.640993788819876,"average_precision":0.609414878081958},{"experiment":"s10_gate","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":25,"fp":18,"fn":2,"tp":13,"recall":0.866666666666667,"specificity":0.581395348837209,"precision":0.419354838709677,"f1":0.565217391304348,"auc":0.852713178294574,"average_precision":0.616122128416246},{"experiment":"s10_gate","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":33,"fp":14,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.702127659574468,"precision":0.333333333333333,"f1":0.4375,"auc":0.738878143133462,"average_precision":0.439709595959596},{"experiment":"s10_gate","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":25,"fp":12,"fn":5,"tp":16,"recall":0.761904761904762,"specificity":0.675675675675676,"precision":0.571428571428571,"f1":0.653061224489796,"auc":0.76962676962677,"average_precision":0.729133222957729},{"experiment":"s10_gate","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.478260869565217,"precision":0.72093023255814,"f1":0.794871794871795,"auc":0.879503105590062,"average_precision":0.932618301662296},{"experiment":"s10_gate","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":17,"fp":14,"fn":4,"tp":23,"recall":0.851851851851852,"specificity":0.548387096774194,"precision":0.621621621621622,"f1":0.71875,"auc":0.765830346475508,"average_precision":0.675838782803418},{"experiment":"s10_gate","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":35,"fp":11,"fn":4,"tp":8,"recall":0.666666666666667,"specificity":0.760869565217391,"precision":0.421052631578947,"f1":0.516129032258064,"auc":0.782608695652174,"average_precision":0.693449243992722},{"experiment":"s10_gate","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":24,"fp":15,"fn":5,"tp":14,"recall":0.736842105263158,"specificity":0.615384615384615,"precision":0.482758620689655,"f1":0.583333333333333,"auc":0.739541160593792,"average_precision":0.57903687548911},{"experiment":"s10_gate","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":32,"fp":8,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.8,"precision":0.466666666666667,"f1":0.424242424242424,"auc":0.658333333333333,"average_precision":0.579447950100934},{"experiment":"s11_cross","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":26,"fp":8,"fn":10,"tp":14,"recall":0.583333333333333,"specificity":0.764705882352941,"precision":0.636363636363636,"f1":0.608695652173913,"auc":0.732843137254902,"average_precision":0.668665479302042},{"experiment":"s11_cross","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":37,"fp":12,"fn":6,"tp":3,"recall":0.333333333333333,"specificity":0.755102040816326,"precision":0.2,"f1":0.25,"auc":0.630385487528345,"average_precision":0.285069310889911},{"experiment":"s11_cross","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":6,"tp":20,"recall":0.769230769230769,"specificity":0.5625,"precision":0.588235294117647,"f1":0.666666666666667,"auc":0.758413461538462,"average_precision":0.721024293669094},{"experiment":"s11_cross","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":28,"fp":7,"fn":17,"tp":6,"recall":0.260869565217391,"specificity":0.8,"precision":0.461538461538462,"f1":0.333333333333333,"auc":0.614906832298137,"average_precision":0.500240370189614},{"experiment":"s11_cross","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":29,"fp":14,"fn":1,"tp":14,"recall":0.933333333333333,"specificity":0.674418604651163,"precision":0.5,"f1":0.651162790697674,"auc":0.866666666666667,"average_precision":0.724515077515078},{"experiment":"s11_cross","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":31,"fp":16,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.659574468085106,"precision":0.304347826086956,"f1":0.411764705882353,"auc":0.692456479690522,"average_precision":0.366780113825568},{"experiment":"s11_cross","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":22,"fp":15,"fn":7,"tp":14,"recall":0.666666666666667,"specificity":0.594594594594595,"precision":0.482758620689655,"f1":0.56,"auc":0.754182754182754,"average_precision":0.631239576265999},{"experiment":"s11_cross","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":9,"fp":14,"fn":1,"tp":34,"recall":0.971428571428571,"specificity":0.391304347826087,"precision":0.708333333333333,"f1":0.819277108433735,"auc":0.850931677018634,"average_precision":0.895679849724555},{"experiment":"s11_cross","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":14,"fp":17,"fn":9,"tp":18,"recall":0.666666666666667,"specificity":0.451612903225806,"precision":0.514285714285714,"f1":0.580645161290323,"auc":0.673835125448029,"average_precision":0.683288101075338},{"experiment":"s11_cross","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":37,"fp":9,"fn":1,"tp":11,"recall":0.916666666666667,"specificity":0.804347826086956,"precision":0.55,"f1":0.6875,"auc":0.903985507246377,"average_precision":0.670961019974178},{"experiment":"s11_cross","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":28,"fp":11,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.717948717948718,"precision":0.541666666666667,"f1":0.604651162790698,"auc":0.754385964912281,"average_precision":0.6853935070197},{"experiment":"s11_cross","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":33,"fp":7,"fn":10,"tp":8,"recall":0.444444444444444,"specificity":0.825,"precision":0.533333333333333,"f1":0.484848484848485,"auc":0.743055555555556,"average_precision":0.636672914667954},{"experiment":"s12_finetune","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":22,"fp":12,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.647058823529412,"precision":0.586206896551724,"f1":0.641509433962264,"auc":0.696078431372549,"average_precision":0.593148936234628},{"experiment":"s12_finetune","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":40,"fp":9,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.816326530612245,"precision":0.181818181818182,"f1":0.2,"auc":0.591836734693878,"average_precision":0.224429891691796},{"experiment":"s12_finetune","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":21,"fp":11,"fn":11,"tp":15,"recall":0.576923076923077,"specificity":0.65625,"precision":0.576923076923077,"f1":0.576923076923077,"auc":0.677884615384615,"average_precision":0.678937265658017},{"experiment":"s12_finetune","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":23,"fp":12,"fn":14,"tp":9,"recall":0.391304347826087,"specificity":0.657142857142857,"precision":0.428571428571428,"f1":0.409090909090909,"auc":0.604968944099379,"average_precision":0.462223968359421},{"experiment":"s12_finetune","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":29,"fp":14,"fn":1,"tp":14,"recall":0.933333333333333,"specificity":0.674418604651163,"precision":0.5,"f1":0.651162790697674,"auc":0.893023255813954,"average_precision":0.786400698067365},{"experiment":"s12_finetune","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":31,"fp":16,"fn":3,"tp":8,"recall":0.727272727272727,"specificity":0.659574468085106,"precision":0.333333333333333,"f1":0.457142857142857,"auc":0.725338491295938,"average_precision":0.336127642411065},{"experiment":"s12_finetune","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":24,"fp":13,"fn":6,"tp":15,"recall":0.714285714285714,"specificity":0.648648648648649,"precision":0.535714285714286,"f1":0.612244897959184,"auc":0.743886743886744,"average_precision":0.75372751357364},{"experiment":"s12_finetune","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":10,"fp":13,"fn":1,"tp":34,"recall":0.971428571428571,"specificity":0.434782608695652,"precision":0.723404255319149,"f1":0.829268292682927,"auc":0.863354037267081,"average_precision":0.896738594663294},{"experiment":"s12_finetune","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":13,"fp":18,"fn":4,"tp":23,"recall":0.851851851851852,"specificity":0.419354838709677,"precision":0.560975609756098,"f1":0.676470588235294,"auc":0.728793309438471,"average_precision":0.699747245843464},{"experiment":"s12_finetune","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":34,"fp":12,"fn":4,"tp":8,"recall":0.666666666666667,"specificity":0.739130434782609,"precision":0.4,"f1":0.5,"auc":0.818840579710145,"average_precision":0.513293188293188},{"experiment":"s12_finetune","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":25,"fp":14,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.641025641025641,"precision":0.481481481481481,"f1":0.565217391304348,"auc":0.693657219973009,"average_precision":0.489207515797721},{"experiment":"s12_finetune","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":35,"fp":5,"fn":12,"tp":6,"recall":0.333333333333333,"specificity":0.875,"precision":0.545454545454545,"f1":0.413793103448276,"auc":0.656944444444444,"average_precision":0.444839608440381},{"experiment":"s13_ensemble","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":25,"fp":9,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.735294117647059,"precision":0.653846153846154,"f1":0.68,"auc":0.758578431372549,"average_precision":0.695595914008424},{"experiment":"s13_ensemble","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":40,"fp":9,"fn":6,"tp":3,"recall":0.333333333333333,"specificity":0.816326530612245,"precision":0.25,"f1":0.285714285714286,"auc":0.612244897959184,"average_precision":0.253027514621718},{"experiment":"s13_ensemble","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":20,"fp":12,"fn":7,"tp":19,"recall":0.730769230769231,"specificity":0.625,"precision":0.612903225806452,"f1":0.666666666666667,"auc":0.747596153846154,"average_precision":0.759133587152076},{"experiment":"s13_ensemble","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":25,"fp":10,"fn":17,"tp":6,"recall":0.260869565217391,"specificity":0.714285714285714,"precision":0.375,"f1":0.307692307692308,"auc":0.617391304347826,"average_precision":0.485219880627754},{"experiment":"s13_ensemble","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":27,"fp":16,"fn":1,"tp":14,"recall":0.933333333333333,"specificity":0.627906976744186,"precision":0.466666666666667,"f1":0.622222222222222,"auc":0.894573643410853,"average_precision":0.807993715100784},{"experiment":"s13_ensemble","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":30,"fp":17,"fn":3,"tp":8,"recall":0.727272727272727,"specificity":0.638297872340426,"precision":0.32,"f1":0.444444444444444,"auc":0.702127659574468,"average_precision":0.37636215537285},{"experiment":"s13_ensemble","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":25,"fp":12,"fn":7,"tp":14,"recall":0.666666666666667,"specificity":0.675675675675676,"precision":0.538461538461538,"f1":0.595744680851064,"auc":0.749034749034749,"average_precision":0.71703295592047},{"experiment":"s13_ensemble","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":10,"fp":13,"fn":0,"tp":35,"recall":1.0,"specificity":0.434782608695652,"precision":0.729166666666667,"f1":0.843373493975904,"auc":0.903105590062112,"average_precision":0.935568408297921},{"experiment":"s13_ensemble","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":14,"fp":17,"fn":8,"tp":19,"recall":0.703703703703704,"specificity":0.451612903225806,"precision":0.527777777777778,"f1":0.603174603174603,"auc":0.710872162485066,"average_precision":0.733048675338124},{"experiment":"s13_ensemble","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":36,"fp":10,"fn":2,"tp":10,"recall":0.833333333333333,"specificity":0.782608695652174,"precision":0.5,"f1":0.625,"auc":0.889492753623188,"average_precision":0.650009497072527},{"experiment":"s13_ensemble","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":26,"fp":13,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.666666666666667,"precision":0.5,"f1":0.577777777777778,"auc":0.754385964912281,"average_precision":0.664641141706934},{"experiment":"s13_ensemble","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":34,"fp":6,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.85,"precision":0.538461538461538,"f1":0.451612903225806,"auc":0.729166666666667,"average_precision":0.56055126797217},{"experiment":"e06_gold_control","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":26,"fp":8,"fn":10,"tp":14,"recall":0.583333333333333,"specificity":0.764705882352941,"precision":0.636363636363636,"f1":0.608695652173913,"auc":0.727941176470588,"average_precision":0.652641878003009},{"experiment":"e06_gold_control","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":47,"fp":2,"fn":9,"tp":0,"recall":0.0,"specificity":0.959183673469388,"precision":0.0,"f1":0.0,"auc":0.424036281179138,"average_precision":0.146582072084918},{"experiment":"e06_gold_control","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":14,"fp":18,"fn":10,"tp":16,"recall":0.615384615384615,"specificity":0.4375,"precision":0.470588235294118,"f1":0.533333333333333,"auc":0.516826923076923,"average_precision":0.52756238134482},{"experiment":"e06_gold_control","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":27,"fp":8,"fn":14,"tp":9,"recall":0.391304347826087,"specificity":0.771428571428572,"precision":0.529411764705882,"f1":0.45,"auc":0.675776397515528,"average_precision":0.525027889365053},{"experiment":"e06_gold_control","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":38,"fp":5,"fn":8,"tp":7,"recall":0.466666666666667,"specificity":0.883720930232558,"precision":0.583333333333333,"f1":0.518518518518518,"auc":0.69922480620155,"average_precision":0.588234541211435},{"experiment":"e06_gold_control","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":46,"fp":1,"fn":9,"tp":2,"recall":0.181818181818182,"specificity":0.978723404255319,"precision":0.666666666666667,"f1":0.285714285714286,"auc":0.655705996131528,"average_precision":0.319896092128455},{"experiment":"e06_gold_control","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":24,"fp":13,"fn":9,"tp":12,"recall":0.571428571428571,"specificity":0.648648648648649,"precision":0.48,"f1":0.521739130434783,"auc":0.625482625482626,"average_precision":0.543452615360468},{"experiment":"e06_gold_control","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":14,"fp":9,"fn":2,"tp":33,"recall":0.942857142857143,"specificity":0.608695652173913,"precision":0.785714285714286,"f1":0.857142857142857,"auc":0.857142857142857,"average_precision":0.888217474174794},{"experiment":"e06_gold_control","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":17,"fp":14,"fn":12,"tp":15,"recall":0.555555555555556,"specificity":0.548387096774194,"precision":0.517241379310345,"f1":0.535714285714286,"auc":0.58183990442055,"average_precision":0.540053854950509},{"experiment":"e06_gold_control","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":36,"fp":10,"fn":12,"tp":0,"recall":0.0,"specificity":0.782608695652174,"precision":0.0,"f1":0.0,"auc":0.278985507246377,"average_precision":0.15030346488576},{"experiment":"e06_gold_control","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":31,"fp":8,"fn":13,"tp":6,"recall":0.31578947368421,"specificity":0.794871794871795,"precision":0.428571428571428,"f1":0.363636363636364,"auc":0.565452091767881,"average_precision":0.426018227272097},{"experiment":"e06_gold_control","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":33,"fp":7,"fn":16,"tp":2,"recall":0.111111111111111,"specificity":0.825,"precision":0.222222222222222,"f1":0.148148148148148,"auc":0.418055555555556,"average_precision":0.280970663779888},{"experiment":"e07_weak_labels","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":25,"fp":9,"fn":11,"tp":13,"recall":0.541666666666667,"specificity":0.735294117647059,"precision":0.590909090909091,"f1":0.565217391304348,"auc":0.698529411764706,"average_precision":0.607761815448378},{"experiment":"e07_weak_labels","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":46,"fp":3,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.938775510204082,"precision":0.4,"f1":0.285714285714286,"auc":0.616780045351474,"average_precision":0.318998160840266},{"experiment":"e07_weak_labels","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":20,"fp":12,"fn":7,"tp":19,"recall":0.730769230769231,"specificity":0.625,"precision":0.612903225806452,"f1":0.666666666666667,"auc":0.745192307692308,"average_precision":0.70450495364441},{"experiment":"e07_weak_labels","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":29,"fp":6,"fn":9,"tp":14,"recall":0.608695652173913,"specificity":0.828571428571429,"precision":0.7,"f1":0.651162790697674,"auc":0.742857142857143,"average_precision":0.697439050682098},{"experiment":"e07_weak_labels","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":29,"fp":14,"fn":2,"tp":13,"recall":0.866666666666667,"specificity":0.674418604651163,"precision":0.481481481481481,"f1":0.619047619047619,"auc":0.875968992248062,"average_precision":0.769642806417},{"experiment":"e07_weak_labels","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":35,"fp":12,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.74468085106383,"precision":0.333333333333333,"f1":0.413793103448276,"auc":0.769825918762089,"average_precision":0.42224233360597},{"experiment":"e07_weak_labels","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":27,"fp":10,"fn":5,"tp":16,"recall":0.761904761904762,"specificity":0.72972972972973,"precision":0.615384615384615,"f1":0.680851063829787,"auc":0.803088803088803,"average_precision":0.78206287146786},{"experiment":"e07_weak_labels","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.478260869565217,"precision":0.72093023255814,"f1":0.794871794871795,"auc":0.862111801242236,"average_precision":0.921495257160783},{"experiment":"e07_weak_labels","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":16,"fp":15,"fn":12,"tp":15,"recall":0.555555555555556,"specificity":0.516129032258064,"precision":0.5,"f1":0.526315789473684,"auc":0.556750298685783,"average_precision":0.592113903191538},{"experiment":"e07_weak_labels","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":31,"fp":15,"fn":3,"tp":9,"recall":0.75,"specificity":0.673913043478261,"precision":0.375,"f1":0.5,"auc":0.78804347826087,"average_precision":0.468768037518038},{"experiment":"e07_weak_labels","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":26,"fp":13,"fn":7,"tp":12,"recall":0.631578947368421,"specificity":0.666666666666667,"precision":0.48,"f1":0.545454545454545,"auc":0.699055330634278,"average_precision":0.555177815432191},{"experiment":"e07_weak_labels","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":32,"fp":8,"fn":11,"tp":7,"recall":0.388888888888889,"specificity":0.8,"precision":0.466666666666667,"f1":0.424242424242424,"auc":0.666666666666667,"average_precision":0.520546377936127},{"experiment":"e08_triplet","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":21,"fp":13,"fn":11,"tp":13,"recall":0.541666666666667,"specificity":0.61764705882353,"precision":0.5,"f1":0.52,"auc":0.682598039215686,"average_precision":0.583913218797971},{"experiment":"e08_triplet","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":41,"fp":8,"fn":6,"tp":3,"recall":0.333333333333333,"specificity":0.836734693877551,"precision":0.272727272727273,"f1":0.3,"auc":0.650793650793651,"average_precision":0.354182935800583},{"experiment":"e08_triplet","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":15,"fp":17,"fn":8,"tp":18,"recall":0.692307692307692,"specificity":0.46875,"precision":0.514285714285714,"f1":0.59016393442623,"auc":0.715144230769231,"average_precision":0.738508535692703},{"experiment":"e08_triplet","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":28,"fp":7,"fn":12,"tp":11,"recall":0.478260869565217,"specificity":0.8,"precision":0.611111111111111,"f1":0.536585365853659,"auc":0.709316770186335,"average_precision":0.644365305756995},{"experiment":"e08_triplet","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":27,"fp":16,"fn":2,"tp":13,"recall":0.866666666666667,"specificity":0.627906976744186,"precision":0.448275862068966,"f1":0.590909090909091,"auc":0.84031007751938,"average_precision":0.712826662177575},{"experiment":"e08_triplet","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":33,"fp":14,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.702127659574468,"precision":0.3,"f1":0.387096774193548,"auc":0.750483558994197,"average_precision":0.455162994578032},{"experiment":"e08_triplet","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":23,"fp":14,"fn":5,"tp":16,"recall":0.761904761904762,"specificity":0.621621621621622,"precision":0.533333333333333,"f1":0.627450980392157,"auc":0.791505791505792,"average_precision":0.730150891326601},{"experiment":"e08_triplet","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":16,"fp":7,"fn":1,"tp":34,"recall":0.971428571428571,"specificity":0.695652173913044,"precision":0.829268292682927,"f1":0.894736842105263,"auc":0.88695652173913,"average_precision":0.921108841033496},{"experiment":"e08_triplet","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":15,"fp":16,"fn":11,"tp":16,"recall":0.592592592592592,"specificity":0.483870967741936,"precision":0.5,"f1":0.542372881355932,"auc":0.556750298685783,"average_precision":0.52812609106113},{"experiment":"e08_triplet","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":34,"fp":12,"fn":2,"tp":10,"recall":0.833333333333333,"specificity":0.739130434782609,"precision":0.454545454545454,"f1":0.588235294117647,"auc":0.827898550724638,"average_precision":0.47436727502517},{"experiment":"e08_triplet","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":31,"fp":8,"fn":7,"tp":12,"recall":0.631578947368421,"specificity":0.794871794871795,"precision":0.6,"f1":0.615384615384615,"auc":0.758434547908232,"average_precision":0.642324308675895},{"experiment":"e08_triplet","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":34,"fp":6,"fn":14,"tp":4,"recall":0.222222222222222,"specificity":0.85,"precision":0.4,"f1":0.285714285714286,"auc":0.719444444444444,"average_precision":0.578519145095737},{"experiment":"e09_protocols","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":26,"fp":8,"fn":13,"tp":11,"recall":0.458333333333333,"specificity":0.764705882352941,"precision":0.578947368421053,"f1":0.511627906976744,"auc":0.662990196078431,"average_precision":0.631377041740765},{"experiment":"e09_protocols","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":40,"fp":9,"fn":6,"tp":3,"recall":0.333333333333333,"specificity":0.816326530612245,"precision":0.25,"f1":0.285714285714286,"auc":0.614512471655329,"average_precision":0.233318446077067},{"experiment":"e09_protocols","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":14,"fp":18,"fn":4,"tp":22,"recall":0.846153846153846,"specificity":0.4375,"precision":0.55,"f1":0.666666666666667,"auc":0.784855769230769,"average_precision":0.774194747801813},{"experiment":"e09_protocols","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":28,"fp":7,"fn":14,"tp":9,"recall":0.391304347826087,"specificity":0.8,"precision":0.5625,"f1":0.461538461538462,"auc":0.647204968944099,"average_precision":0.583073077564168},{"experiment":"e09_protocols","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":1,"tp":14,"recall":0.933333333333333,"specificity":0.697674418604651,"precision":0.518518518518518,"f1":0.666666666666667,"auc":0.872868217054264,"average_precision":0.747158348897479},{"experiment":"e09_protocols","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":33,"fp":14,"fn":3,"tp":8,"recall":0.727272727272727,"specificity":0.702127659574468,"precision":0.363636363636364,"f1":0.484848484848485,"auc":0.798839458413926,"average_precision":0.500388742847657},{"experiment":"e09_protocols","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":21,"fp":16,"fn":3,"tp":18,"recall":0.857142857142857,"specificity":0.567567567567568,"precision":0.529411764705882,"f1":0.654545454545454,"auc":0.812097812097812,"average_precision":0.744997808828161},{"experiment":"e09_protocols","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":3,"tp":32,"recall":0.914285714285714,"specificity":0.478260869565217,"precision":0.727272727272727,"f1":0.810126582278481,"auc":0.809937888198758,"average_precision":0.85052543406406},{"experiment":"e09_protocols","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":17,"fp":14,"fn":9,"tp":18,"recall":0.666666666666667,"specificity":0.548387096774194,"precision":0.5625,"f1":0.610169491525424,"auc":0.664277180406213,"average_precision":0.602644961436818},{"experiment":"e09_protocols","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":32,"fp":14,"fn":1,"tp":11,"recall":0.916666666666667,"specificity":0.695652173913044,"precision":0.44,"f1":0.594594594594595,"auc":0.807971014492754,"average_precision":0.433923810752371},{"experiment":"e09_protocols","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":26,"fp":13,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.666666666666667,"precision":0.5,"f1":0.577777777777778,"auc":0.712550607287449,"average_precision":0.550500048072364},{"experiment":"e09_protocols","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":34,"fp":6,"fn":12,"tp":6,"recall":0.333333333333333,"specificity":0.85,"precision":0.5,"f1":0.4,"auc":0.744444444444444,"average_precision":0.581183425409464},{"experiment":"e10_class_mil","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":22,"fp":12,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.647058823529412,"precision":0.586206896551724,"f1":0.641509433962264,"auc":0.704656862745098,"average_precision":0.665732548597353},{"experiment":"e10_class_mil","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":41,"fp":8,"fn":6,"tp":3,"recall":0.333333333333333,"specificity":0.836734693877551,"precision":0.272727272727273,"f1":0.3,"auc":0.580498866213152,"average_precision":0.220386210028889},{"experiment":"e10_class_mil","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":21,"fp":11,"fn":10,"tp":16,"recall":0.615384615384615,"specificity":0.65625,"precision":0.592592592592592,"f1":0.60377358490566,"auc":0.670673076923077,"average_precision":0.621372982366106},{"experiment":"e10_class_mil","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":26,"fp":9,"fn":17,"tp":6,"recall":0.260869565217391,"specificity":0.742857142857143,"precision":0.4,"f1":0.31578947368421,"auc":0.556521739130435,"average_precision":0.48801736973086},{"experiment":"e10_class_mil","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":4,"tp":11,"recall":0.733333333333333,"specificity":0.697674418604651,"precision":0.458333333333333,"f1":0.564102564102564,"auc":0.834108527131783,"average_precision":0.702405843640364},{"experiment":"e10_class_mil","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":36,"fp":11,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.765957446808511,"precision":0.352941176470588,"f1":0.428571428571428,"auc":0.630560928433269,"average_precision":0.401548549489726},{"experiment":"e10_class_mil","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":23,"fp":14,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.621621621621622,"precision":0.548387096774194,"f1":0.653846153846154,"auc":0.796653796653797,"average_precision":0.790091297327305},{"experiment":"e10_class_mil","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":12,"fp":11,"fn":4,"tp":31,"recall":0.885714285714286,"specificity":0.521739130434783,"precision":0.738095238095238,"f1":0.805194805194805,"auc":0.801242236024845,"average_precision":0.876655206305732},{"experiment":"e10_class_mil","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":18,"fp":13,"fn":8,"tp":19,"recall":0.703703703703704,"specificity":0.580645161290323,"precision":0.59375,"f1":0.64406779661017,"auc":0.695340501792115,"average_precision":0.628200325672058},{"experiment":"e10_class_mil","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":32,"fp":14,"fn":6,"tp":6,"recall":0.5,"specificity":0.695652173913044,"precision":0.3,"f1":0.375,"auc":0.740942028985507,"average_precision":0.494637884839544},{"experiment":"e10_class_mil","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":23,"fp":16,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.58974358974359,"precision":0.448275862068966,"f1":0.541666666666667,"auc":0.738191632928475,"average_precision":0.576808015193995},{"experiment":"e10_class_mil","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":37,"fp":3,"fn":12,"tp":6,"recall":0.333333333333333,"specificity":0.925,"precision":0.666666666666667,"f1":0.444444444444444,"auc":0.740277777777778,"average_precision":0.598310373603575},{"experiment":"e11_finetune","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":20,"fp":14,"fn":8,"tp":16,"recall":0.666666666666667,"specificity":0.588235294117647,"precision":0.533333333333333,"f1":0.592592592592592,"auc":0.675245098039216,"average_precision":0.667654873030257},{"experiment":"e11_finetune","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":42,"fp":7,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.857142857142857,"precision":0.222222222222222,"f1":0.222222222222222,"auc":0.489795918367347,"average_precision":0.210718012054923},{"experiment":"e11_finetune","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":21,"fp":11,"fn":8,"tp":18,"recall":0.692307692307692,"specificity":0.65625,"precision":0.620689655172414,"f1":0.654545454545454,"auc":0.700721153846154,"average_precision":0.633023124321296},{"experiment":"e11_finetune","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":24,"fp":11,"fn":15,"tp":8,"recall":0.347826086956522,"specificity":0.685714285714286,"precision":0.421052631578947,"f1":0.380952380952381,"auc":0.606211180124224,"average_precision":0.539072800959632},{"experiment":"e11_finetune","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":30,"fp":13,"fn":5,"tp":10,"recall":0.666666666666667,"specificity":0.697674418604651,"precision":0.434782608695652,"f1":0.526315789473684,"auc":0.793798449612403,"average_precision":0.636358852649175},{"experiment":"e11_finetune","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":29,"fp":18,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.617021276595745,"precision":0.25,"f1":0.342857142857143,"auc":0.622823984526112,"average_precision":0.524220529915717},{"experiment":"e11_finetune","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":26,"fp":11,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.702702702702703,"precision":0.607142857142857,"f1":0.693877551020408,"auc":0.836550836550836,"average_precision":0.824296269637885},{"experiment":"e11_finetune","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":7,"tp":28,"recall":0.8,"specificity":0.478260869565217,"precision":0.7,"f1":0.746666666666667,"auc":0.762732919254658,"average_precision":0.834968572715282},{"experiment":"e11_finetune","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":17,"fp":14,"fn":9,"tp":18,"recall":0.666666666666667,"specificity":0.548387096774194,"precision":0.5625,"f1":0.610169491525424,"auc":0.654719235364397,"average_precision":0.589540904527484},{"experiment":"e11_finetune","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":36,"fp":10,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.782608695652174,"precision":0.411764705882353,"f1":0.482758620689655,"auc":0.807971014492754,"average_precision":0.457601060072324},{"experiment":"e11_finetune","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":26,"fp":13,"fn":4,"tp":15,"recall":0.789473684210526,"specificity":0.666666666666667,"precision":0.535714285714286,"f1":0.638297872340426,"auc":0.807017543859649,"average_precision":0.628774984926642},{"experiment":"e11_finetune","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":38,"fp":2,"fn":12,"tp":6,"recall":0.333333333333333,"specificity":0.95,"precision":0.75,"f1":0.461538461538462,"auc":0.711111111111111,"average_precision":0.595723911157924},{"experiment":"e12_ensemble","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":22,"fp":12,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.647058823529412,"precision":0.586206896551724,"f1":0.641509433962264,"auc":0.700980392156863,"average_precision":0.683998455021739},{"experiment":"e12_ensemble","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":42,"fp":7,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.857142857142857,"precision":0.222222222222222,"f1":0.222222222222222,"auc":0.532879818594104,"average_precision":0.208439988043803},{"experiment":"e12_ensemble","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":21,"fp":11,"fn":9,"tp":17,"recall":0.653846153846154,"specificity":0.65625,"precision":0.607142857142857,"f1":0.62962962962963,"auc":0.697115384615385,"average_precision":0.648907638972511},{"experiment":"e12_ensemble","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":27,"fp":8,"fn":17,"tp":6,"recall":0.260869565217391,"specificity":0.771428571428572,"precision":0.428571428571428,"f1":0.324324324324324,"auc":0.587577639751553,"average_precision":0.515741861431828},{"experiment":"e12_ensemble","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":29,"fp":14,"fn":4,"tp":11,"recall":0.733333333333333,"specificity":0.674418604651163,"precision":0.44,"f1":0.55,"auc":0.817054263565892,"average_precision":0.674490710577667},{"experiment":"e12_ensemble","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":29,"fp":18,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.617021276595745,"precision":0.25,"f1":0.342857142857143,"auc":0.638297872340426,"average_precision":0.487011970364505},{"experiment":"e12_ensemble","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":23,"fp":14,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.621621621621622,"precision":0.548387096774194,"f1":0.653846153846154,"auc":0.826254826254826,"average_precision":0.813624177906846},{"experiment":"e12_ensemble","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":11,"fp":12,"fn":6,"tp":29,"recall":0.828571428571429,"specificity":0.478260869565217,"precision":0.707317073170732,"f1":0.763157894736842,"auc":0.788819875776398,"average_precision":0.860383539737654},{"experiment":"e12_ensemble","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":19,"fp":12,"fn":8,"tp":19,"recall":0.703703703703704,"specificity":0.612903225806452,"precision":0.612903225806452,"f1":0.655172413793103,"auc":0.67741935483871,"average_precision":0.60752597055598},{"experiment":"e12_ensemble","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":35,"fp":11,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.760869565217391,"precision":0.388888888888889,"f1":0.466666666666667,"auc":0.784420289855072,"average_precision":0.488846466317106},{"experiment":"e12_ensemble","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":24,"fp":15,"fn":4,"tp":15,"recall":0.789473684210526,"specificity":0.615384615384615,"precision":0.5,"f1":0.612244897959184,"auc":0.784075573549258,"average_precision":0.621664032350195},{"experiment":"e12_ensemble","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":37,"fp":3,"fn":12,"tp":6,"recall":0.333333333333333,"specificity":0.925,"precision":0.666666666666667,"f1":0.444444444444444,"auc":0.729166666666667,"average_precision":0.598873872146819},{"experiment":"e00_resnet18_3","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":15,"fp":19,"fn":10,"tp":14,"recall":0.583333333333333,"specificity":0.441176470588235,"precision":0.424242424242424,"f1":0.491228070175438,"auc":0.503676470588235,"average_precision":0.477503205038064},{"experiment":"e00_resnet18_3","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":24,"fp":25,"fn":4,"tp":5,"recall":0.555555555555556,"specificity":0.489795918367347,"precision":0.166666666666667,"f1":0.256410256410256,"auc":0.53968253968254,"average_precision":0.220800064133397},{"experiment":"e00_resnet18_3","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":11,"tp":15,"recall":0.576923076923077,"specificity":0.5625,"precision":0.517241379310345,"f1":0.545454545454545,"auc":0.598557692307692,"average_precision":0.557191432962595},{"experiment":"e00_resnet18_3","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":25,"fp":10,"fn":8,"tp":15,"recall":0.652173913043478,"specificity":0.714285714285714,"precision":0.6,"f1":0.625,"auc":0.693167701863354,"average_precision":0.557987917395895},{"experiment":"e00_resnet18_3","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":22,"fp":21,"fn":6,"tp":9,"recall":0.6,"specificity":0.511627906976744,"precision":0.3,"f1":0.4,"auc":0.654263565891473,"average_precision":0.417665206927533},{"experiment":"e00_resnet18_3","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":29,"fp":18,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.617021276595745,"precision":0.28,"f1":0.388888888888889,"auc":0.609284332688588,"average_precision":0.297909545310659},{"experiment":"e00_resnet18_3","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":15,"fp":22,"fn":6,"tp":15,"recall":0.714285714285714,"specificity":0.405405405405405,"precision":0.405405405405405,"f1":0.517241379310345,"auc":0.546975546975547,"average_precision":0.439599039708755},{"experiment":"e00_resnet18_3","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":12,"fp":11,"fn":11,"tp":24,"recall":0.685714285714286,"specificity":0.521739130434783,"precision":0.685714285714286,"f1":0.685714285714286,"auc":0.536645962732919,"average_precision":0.620310489276229},{"experiment":"e00_resnet18_3","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":16,"fp":15,"fn":11,"tp":16,"recall":0.592592592592592,"specificity":0.516129032258064,"precision":0.516129032258064,"f1":0.551724137931034,"auc":0.565113500597372,"average_precision":0.655489898246713},{"experiment":"e00_resnet18_3","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":25,"fp":21,"fn":4,"tp":8,"recall":0.666666666666667,"specificity":0.543478260869565,"precision":0.275862068965517,"f1":0.390243902439024,"auc":0.492753623188406,"average_precision":0.214655707422046},{"experiment":"e00_resnet18_3","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":22,"fp":17,"fn":6,"tp":13,"recall":0.68421052631579,"specificity":0.564102564102564,"precision":0.433333333333333,"f1":0.530612244897959,"auc":0.662618083670715,"average_precision":0.575597765999807},{"experiment":"e00_resnet18_3","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":25,"fp":15,"fn":6,"tp":12,"recall":0.666666666666667,"specificity":0.625,"precision":0.444444444444444,"f1":0.533333333333333,"auc":0.597222222222222,"average_precision":0.430762541016959},{"experiment":"e01_resnet18_5","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":16,"fp":18,"fn":12,"tp":12,"recall":0.5,"specificity":0.470588235294118,"precision":0.4,"f1":0.444444444444444,"auc":0.547794117647059,"average_precision":0.544177643157806},{"experiment":"e01_resnet18_5","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":26,"fp":23,"fn":4,"tp":5,"recall":0.555555555555556,"specificity":0.530612244897959,"precision":0.178571428571428,"f1":0.27027027027027,"auc":0.489795918367347,"average_precision":0.201764081451044},{"experiment":"e01_resnet18_5","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":11,"tp":15,"recall":0.576923076923077,"specificity":0.5625,"precision":0.517241379310345,"f1":0.545454545454545,"auc":0.572115384615385,"average_precision":0.5045248353359},{"experiment":"e01_resnet18_5","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":22,"fp":13,"fn":8,"tp":15,"recall":0.652173913043478,"specificity":0.628571428571428,"precision":0.535714285714286,"f1":0.588235294117647,"auc":0.672049689440994,"average_precision":0.558313238223077},{"experiment":"e01_resnet18_5","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":18,"fp":25,"fn":4,"tp":11,"recall":0.733333333333333,"specificity":0.418604651162791,"precision":0.305555555555556,"f1":0.431372549019608,"auc":0.665116279069768,"average_precision":0.442222575081014},{"experiment":"e01_resnet18_5","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":28,"fp":19,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.595744680851064,"precision":0.269230769230769,"f1":0.378378378378378,"auc":0.62862669245648,"average_precision":0.307776268261728},{"experiment":"e01_resnet18_5","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":22,"fp":15,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.594594594594595,"precision":0.53125,"f1":0.641509433962264,"auc":0.64993564993565,"average_precision":0.528543169647938},{"experiment":"e01_resnet18_5","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":13,"fp":10,"fn":12,"tp":23,"recall":0.657142857142857,"specificity":0.565217391304348,"precision":0.696969696969697,"f1":0.676470588235294,"auc":0.606211180124224,"average_precision":0.665487519359443},{"experiment":"e01_resnet18_5","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":18,"fp":13,"fn":12,"tp":15,"recall":0.555555555555556,"specificity":0.580645161290323,"precision":0.535714285714286,"f1":0.545454545454545,"auc":0.592592592592592,"average_precision":0.665905481905252},{"experiment":"e01_resnet18_5","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":23,"fp":23,"fn":4,"tp":8,"recall":0.666666666666667,"specificity":0.5,"precision":0.258064516129032,"f1":0.372093023255814,"auc":0.483695652173913,"average_precision":0.207566580914539},{"experiment":"e01_resnet18_5","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":20,"fp":19,"fn":5,"tp":14,"recall":0.736842105263158,"specificity":0.512820512820513,"precision":0.424242424242424,"f1":0.538461538461538,"auc":0.674763832658569,"average_precision":0.514831431915731},{"experiment":"e01_resnet18_5","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":24,"fp":16,"fn":8,"tp":10,"recall":0.555555555555556,"specificity":0.6,"precision":0.384615384615385,"f1":0.454545454545454,"auc":0.595833333333333,"average_precision":0.419164359898099},{"experiment":"e02a_dinov2_5","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":22,"fp":12,"fn":9,"tp":15,"recall":0.625,"specificity":0.647058823529412,"precision":0.555555555555556,"f1":0.588235294117647,"auc":0.738970588235294,"average_precision":0.689263944832942},{"experiment":"e02a_dinov2_5","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":26,"fp":23,"fn":6,"tp":3,"recall":0.333333333333333,"specificity":0.530612244897959,"precision":0.115384615384615,"f1":0.171428571428571,"auc":0.458049886621315,"average_precision":0.156540714526223},{"experiment":"e02a_dinov2_5","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":12,"fp":20,"fn":11,"tp":15,"recall":0.576923076923077,"specificity":0.375,"precision":0.428571428571428,"f1":0.491803278688525,"auc":0.527644230769231,"average_precision":0.534314126050813},{"experiment":"e02a_dinov2_5","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":21,"fp":14,"fn":6,"tp":17,"recall":0.739130434782609,"specificity":0.6,"precision":0.548387096774194,"f1":0.62962962962963,"auc":0.677018633540373,"average_precision":0.537863568058396},{"experiment":"e02a_dinov2_5","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":20,"fp":23,"fn":6,"tp":9,"recall":0.6,"specificity":0.465116279069767,"precision":0.28125,"f1":0.382978723404255,"auc":0.593798449612403,"average_precision":0.458328166137858},{"experiment":"e02a_dinov2_5","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":33,"fp":14,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.702127659574468,"precision":0.333333333333333,"f1":0.4375,"auc":0.717601547388781,"average_precision":0.331598051574643},{"experiment":"e02a_dinov2_5","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":19,"fp":18,"fn":8,"tp":13,"recall":0.619047619047619,"specificity":0.513513513513514,"precision":0.419354838709677,"f1":0.5,"auc":0.619047619047619,"average_precision":0.515177610371116},{"experiment":"e02a_dinov2_5","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":15,"fp":8,"fn":13,"tp":22,"recall":0.628571428571428,"specificity":0.652173913043478,"precision":0.733333333333333,"f1":0.676923076923077,"auc":0.742857142857143,"average_precision":0.838244371697864},{"experiment":"e02a_dinov2_5","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":19,"fp":12,"fn":13,"tp":14,"recall":0.518518518518518,"specificity":0.612903225806452,"precision":0.538461538461538,"f1":0.528301886792453,"auc":0.566308243727599,"average_precision":0.511488809749564},{"experiment":"e02a_dinov2_5","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":20,"fp":26,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.434782608695652,"precision":0.212121212121212,"f1":0.311111111111111,"auc":0.472826086956522,"average_precision":0.210722637655807},{"experiment":"e02a_dinov2_5","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":20,"fp":19,"fn":8,"tp":11,"recall":0.578947368421053,"specificity":0.512820512820513,"precision":0.366666666666667,"f1":0.448979591836735,"auc":0.535762483130904,"average_precision":0.382831837851123},{"experiment":"e02a_dinov2_5","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":13,"fp":27,"fn":13,"tp":5,"recall":0.277777777777778,"specificity":0.325,"precision":0.15625,"f1":0.2,"auc":0.286111111111111,"average_precision":0.233545061782685},{"experiment":"e02b_swin_5","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":19,"fp":15,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.558823529411765,"precision":0.53125,"f1":0.607142857142857,"auc":0.699754901960784,"average_precision":0.675825713913922},{"experiment":"e02b_swin_5","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":31,"fp":18,"fn":4,"tp":5,"recall":0.555555555555556,"specificity":0.63265306122449,"precision":0.217391304347826,"f1":0.3125,"auc":0.503401360544218,"average_precision":0.17566473254879},{"experiment":"e02b_swin_5","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":18,"fp":14,"fn":9,"tp":17,"recall":0.653846153846154,"specificity":0.5625,"precision":0.548387096774194,"f1":0.596491228070175,"auc":0.545673076923077,"average_precision":0.532614045778244},{"experiment":"e02b_swin_5","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":22,"fp":13,"fn":5,"tp":18,"recall":0.782608695652174,"specificity":0.628571428571428,"precision":0.580645161290323,"f1":0.666666666666667,"auc":0.693167701863354,"average_precision":0.540729634817079},{"experiment":"e02b_swin_5","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":20,"fp":23,"fn":3,"tp":12,"recall":0.8,"specificity":0.465116279069767,"precision":0.342857142857143,"f1":0.48,"auc":0.697674418604651,"average_precision":0.506952506669603},{"experiment":"e02b_swin_5","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":29,"fp":18,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.617021276595745,"precision":0.25,"f1":0.342857142857143,"auc":0.657640232108317,"average_precision":0.311460452987222},{"experiment":"e02b_swin_5","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":21,"fp":16,"fn":4,"tp":17,"recall":0.80952380952381,"specificity":0.567567567567568,"precision":0.515151515151515,"f1":0.62962962962963,"auc":0.648648648648649,"average_precision":0.558508122555259},{"experiment":"e02b_swin_5","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":17,"fp":6,"fn":7,"tp":28,"recall":0.8,"specificity":0.739130434782609,"precision":0.823529411764706,"f1":0.811594202898551,"auc":0.832298136645963,"average_precision":0.875912366758045},{"experiment":"e02b_swin_5","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":15,"fp":16,"fn":9,"tp":18,"recall":0.666666666666667,"specificity":0.483870967741936,"precision":0.529411764705882,"f1":0.59016393442623,"auc":0.567502986857826,"average_precision":0.522387704540289},{"experiment":"e02b_swin_5","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":21,"fp":25,"fn":7,"tp":5,"recall":0.416666666666667,"specificity":0.456521739130435,"precision":0.166666666666667,"f1":0.238095238095238,"auc":0.369565217391304,"average_precision":0.186183318907501},{"experiment":"e02b_swin_5","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":19,"fp":20,"fn":5,"tp":14,"recall":0.736842105263158,"specificity":0.487179487179487,"precision":0.411764705882353,"f1":0.528301886792453,"auc":0.654520917678812,"average_precision":0.505035364245891},{"experiment":"e02b_swin_5","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":20,"fp":20,"fn":8,"tp":10,"recall":0.555555555555556,"specificity":0.5,"precision":0.333333333333333,"f1":0.416666666666667,"auc":0.572222222222222,"average_precision":0.34339862526137},{"experiment":"e02c_convnext_5","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":22,"fp":12,"fn":9,"tp":15,"recall":0.625,"specificity":0.647058823529412,"precision":0.555555555555556,"f1":0.588235294117647,"auc":0.638480392156863,"average_precision":0.51256234600819},{"experiment":"e02c_convnext_5","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":27,"fp":22,"fn":7,"tp":2,"recall":0.222222222222222,"specificity":0.551020408163265,"precision":0.083333333333333,"f1":0.121212121212121,"auc":0.46031746031746,"average_precision":0.156383022287977},{"experiment":"e02c_convnext_5","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":16,"fp":16,"fn":10,"tp":16,"recall":0.615384615384615,"specificity":0.5,"precision":0.5,"f1":0.551724137931034,"auc":0.501201923076923,"average_precision":0.484724376844053},{"experiment":"e02c_convnext_5","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":20,"fp":15,"fn":9,"tp":14,"recall":0.608695652173913,"specificity":0.571428571428571,"precision":0.482758620689655,"f1":0.538461538461538,"auc":0.638509316770186,"average_precision":0.527107804582061},{"experiment":"e02c_convnext_5","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":25,"fp":18,"fn":1,"tp":14,"recall":0.933333333333333,"specificity":0.581395348837209,"precision":0.4375,"f1":0.595744680851064,"auc":0.787596899224806,"average_precision":0.62600049856623},{"experiment":"e02c_convnext_5","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":29,"fp":18,"fn":4,"tp":7,"recall":0.636363636363636,"specificity":0.617021276595745,"precision":0.28,"f1":0.388888888888889,"auc":0.680851063829787,"average_precision":0.390826103930553},{"experiment":"e02c_convnext_5","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":21,"fp":16,"fn":8,"tp":13,"recall":0.619047619047619,"specificity":0.567567567567568,"precision":0.448275862068966,"f1":0.52,"auc":0.643500643500644,"average_precision":0.621276588151439},{"experiment":"e02c_convnext_5","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":17,"fp":6,"fn":11,"tp":24,"recall":0.685714285714286,"specificity":0.739130434782609,"precision":0.8,"f1":0.738461538461538,"auc":0.804968944099379,"average_precision":0.860032053637497},{"experiment":"e02c_convnext_5","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":16,"fp":15,"fn":12,"tp":15,"recall":0.555555555555556,"specificity":0.516129032258064,"precision":0.5,"f1":0.526315789473684,"auc":0.590203106332139,"average_precision":0.574315476751579},{"experiment":"e02c_convnext_5","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":20,"fp":26,"fn":6,"tp":6,"recall":0.5,"specificity":0.434782608695652,"precision":0.1875,"f1":0.272727272727273,"auc":0.394927536231884,"average_precision":0.187394594140296},{"experiment":"e02c_convnext_5","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":20,"fp":19,"fn":7,"tp":12,"recall":0.631578947368421,"specificity":0.512820512820513,"precision":0.387096774193548,"f1":0.48,"auc":0.62753036437247,"average_precision":0.475177324438842},{"experiment":"e02c_convnext_5","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":20,"fp":20,"fn":9,"tp":9,"recall":0.5,"specificity":0.5,"precision":0.310344827586207,"f1":0.382978723404255,"auc":0.451388888888889,"average_precision":0.278881891936932},{"experiment":"e02d_vit_5","seed":42,"label":"ACL","positives":24,"negatives":34,"tn":22,"fp":12,"fn":7,"tp":17,"recall":0.708333333333333,"specificity":0.647058823529412,"precision":0.586206896551724,"f1":0.641509433962264,"auc":0.768382352941176,"average_precision":0.678275684081127},{"experiment":"e02d_vit_5","seed":42,"label":"MCL","positives":9,"negatives":49,"tn":24,"fp":25,"fn":4,"tp":5,"recall":0.555555555555556,"specificity":0.489795918367347,"precision":0.166666666666667,"f1":0.256410256410256,"auc":0.526077097505669,"average_precision":0.262332915622389},{"experiment":"e02d_vit_5","seed":42,"label":"Medial Meniscus","positives":26,"negatives":32,"tn":17,"fp":15,"fn":9,"tp":17,"recall":0.653846153846154,"specificity":0.53125,"precision":0.53125,"f1":0.586206896551724,"auc":0.573317307692308,"average_precision":0.554992788959788},{"experiment":"e02d_vit_5","seed":42,"label":"Lateral Meniscus","positives":23,"negatives":35,"tn":19,"fp":16,"fn":4,"tp":19,"recall":0.826086956521739,"specificity":0.542857142857143,"precision":0.542857142857143,"f1":0.655172413793103,"auc":0.662111801242236,"average_precision":0.547670006787561},{"experiment":"e02d_vit_5","seed":42,"label":"Medial OA","positives":15,"negatives":43,"tn":16,"fp":27,"fn":4,"tp":11,"recall":0.733333333333333,"specificity":0.372093023255814,"precision":0.289473684210526,"f1":0.415094339622642,"auc":0.582945736434109,"average_precision":0.407620765513948},{"experiment":"e02d_vit_5","seed":42,"label":"Lateral OA","positives":11,"negatives":47,"tn":31,"fp":16,"fn":5,"tp":6,"recall":0.545454545454545,"specificity":0.659574468085106,"precision":0.272727272727273,"f1":0.363636363636364,"auc":0.576402321083172,"average_precision":0.328095480285718},{"experiment":"e02d_vit_5","seed":42,"label":"PF OA","positives":21,"negatives":37,"tn":19,"fp":18,"fn":6,"tp":15,"recall":0.714285714285714,"specificity":0.513513513513514,"precision":0.454545454545454,"f1":0.555555555555556,"auc":0.626769626769627,"average_precision":0.572421945042279},{"experiment":"e02d_vit_5","seed":42,"label":"Effusion","positives":35,"negatives":23,"tn":15,"fp":8,"fn":8,"tp":27,"recall":0.771428571428572,"specificity":0.652173913043478,"precision":0.771428571428572,"f1":0.771428571428572,"auc":0.765217391304348,"average_precision":0.827639976087342},{"experiment":"e02d_vit_5","seed":42,"label":"Synovitis","positives":27,"negatives":31,"tn":18,"fp":13,"fn":14,"tp":13,"recall":0.481481481481481,"specificity":0.580645161290323,"precision":0.5,"f1":0.490566037735849,"auc":0.579450418160096,"average_precision":0.560336211056455},{"experiment":"e02d_vit_5","seed":42,"label":"Baker\'s","positives":12,"negatives":46,"tn":17,"fp":29,"fn":5,"tp":7,"recall":0.583333333333333,"specificity":0.369565217391304,"precision":0.194444444444444,"f1":0.291666666666667,"auc":0.409420289855072,"average_precision":0.183869776946503},{"experiment":"e02d_vit_5","seed":42,"label":"Contusion","positives":19,"negatives":39,"tn":17,"fp":22,"fn":7,"tp":12,"recall":0.631578947368421,"specificity":0.435897435897436,"precision":0.352941176470588,"f1":0.452830188679245,"auc":0.565452091767881,"average_precision":0.419963034857957},{"experiment":"e02d_vit_5","seed":42,"label":"Fracture","positives":18,"negatives":40,"tn":17,"fp":23,"fn":7,"tp":11,"recall":0.611111111111111,"specificity":0.425,"precision":0.323529411764706,"f1":0.423076923076923,"auc":0.523611111111111,"average_precision":0.354521570567718}]'))
assert len(stats) == 31 and len(category) == 372
assert (category[['tn','fp','fn','tp']].sum(axis=1) == 58).all()
display(stats[['experiment','macro_auc','seed_sd','bce','training_studies']].round(6))
stats.to_csv(OUT / 'experiment_summary.csv', index=False)
category.to_csv(OUT / 'category_metrics_seed42.csv', index=False)

## 4. What changed, and what happened?

| Comparison | Mean OOF AUC | Interpretation within this development set |
|---|---|---|
| E06 gold → E07 weak labels | .585972 → .741941 | Largest improvement in this controlled comparison |
| E07 → E08 triplet → E09 protocol slots | .741941 → .742926 → .744113 | Small incremental changes; uncertainty matters |
| S01 neighbour3 → S02 neighbour5 → S03 neighbour10 | .716563 → .732541 → .724005 | Five won under this setting; not a universal optimal slice count |
| S02 → S04 sequence attention → S05 additive position | .732541 → .726607 → .713061 | More complex heads did not automatically help |
| S06 index RoPE → S07 physical-mm RoPE | .721530 → .711794 | Position encoding needs an effective representation and validation |
| S09 MLP → S10 gate / S11 cross | .724314 → .739986 / .736386 | Class-aware fusion was promising; cross did not beat gate |
| S11 → S12 limited fine-tuning | .736386 → .718888 | Current one-epoch setup underperformed; not proof against all fine-tuning |
| S13 → R01 | .748420 → .771432 | Complementary pathways helped in the development OOF |

R01 combines **0.5 E09 + 0.25 S11 + 0.25 S12**, using probabilities. It is not the probability average of three randomly named backbones. S12 can be a useful ensemble member even when its standalone score is weaker; this must be measured, not assumed.

S02→S03 actually changed unique image tokens from 409,239 to 526,504 (+28.7%) and feature extraction from 4.60h to 5.52h. Possible causes of the lower score include redundancy, extra background, model/data complexity, training settings, and sampling variation. These are hypotheses, not proven causes.

In [ ]:
ordered = stats.sort_values('macro_auc')
fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#cf6b36' if e.startswith('r01') else '#246d8c' for e in ordered.experiment]
ax.barh(ordered.experiment, ordered.macro_auc, xerr=ordered.seed_sd, color=colors, alpha=.9)
ax.set(xlim=(.45,.82), xlabel='Mean Gold58 OOF macro ROC-AUC (error bar: seed SD)',
       title='31 completed configurations — same development evaluation')
ax.grid(axis='x', alpha=.2)
fig.tight_layout(); fig.savefig(OUT/'experiment_auc.png', dpi=160); plt.show()

## 5. Confusion matrices: where do we miss findings?

각 소견별 **2×2 행렬**을 사용합니다. 하나의 12×12 혼동행렬을 쓰면 동시에 여러 소견이 가능한 문제를 잘못 표현합니다.

| | Predicted absent | Predicted present |
|---|---|---|
| Actually absent | TN: 정상 소견을 없다고 판단 | FP: 없는 소견을 있다고 판단 |
| Actually present | FN: 있는 소견을 놓침 | TP: 있는 소견을 맞힘 |

- **Recall / 재현율 = TP / (TP+FN)**: 실제 소견이 있는 study 중 얼마나 찾아냈는가? FN이 늘면 내려갑니다.
- **Precision / 정밀도 = TP / (TP+FP)**: 있다고 예측한 것 중 실제로 맞은 비율은 얼마인가? FP가 늘면 내려갑니다.
- TN이 많으면 accuracy가 높아도 양성 소견을 놓칠 수 있습니다. 희귀 소견은 양성 수가 작아 몇 건 차이가 recall을 크게 바꿉니다.
- AUC는 threshold 하나에 고정되지 않습니다. 따라서 AUC가 좋아도 threshold .5의 recall이 낮을 수 있습니다. threshold 선택은 별도 validation에서 해야 합니다.

아래는 seed42 OOF, threshold=.5입니다. S13→R01의 전체 FN은75→67, FP는144→132로 줄었지만 **MCL FN은6→7, Fracture FN은11→13**으로 늘었습니다. 평균 개선과 소견별 개선은 다릅니다.

In [ ]:
def compare_experiments(before, after):
    fields = ['tn','fp','fn','tp','recall','precision','auc']
    a = category.loc[category.experiment.eq(before)].set_index('label')[fields]
    b = category.loc[category.experiment.eq(after)].set_index('label')[fields]
    assert len(a) == len(b) == 12
    result = a.add_suffix('_before').join(b.add_suffix('_after'))
    result['delta_fn'] = result.fn_after - result.fn_before
    result['delta_fp'] = result.fp_after - result.fp_before
    display(result.round(4))
    return result

comparison = compare_experiments('s13_ensemble', 'r01_e09_s13_blend')
fig, axes = plt.subplots(3, 4, figsize=(13, 9))
rows = category.loc[category.experiment.eq('r01_e09_s13_blend')]
for ax, row in zip(axes.flat, rows.itertuples()):
    matrix = np.array([[row.tn,row.fp],[row.fn,row.tp]])
    ax.imshow(matrix, cmap='Blues', vmin=0, vmax=58)
    for r in range(2):
        for c in range(2):
            name = [['TN','FP'],['FN','TP']][r][c]
            ax.text(c,r,f'{name} {matrix[r,c]}',ha='center',va='center',color='black')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Pred -','Pred +'],yticklabels=['True -','True +'],
           title=f'{row.label}\nRecall {row.recall:.2f} | Precision {row.precision:.2f}')
fig.suptitle('R01: 12 binary confusion matrices — Gold58 OOF, seed42, threshold 0.5')
fig.tight_layout(); fig.savefig(OUT/'r01_confusions.png', dpi=160); plt.show()

fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax, field in zip(axes,['fn','fp']):
    comparison[[field+'_before',field+'_after']].plot.barh(ax=ax,color=['#84949f','#cf6b36'])
    ax.set_title(field.upper() + ': S13 vs R01'); ax.set_xlabel('Number of studies'); ax.grid(axis='x',alpha=.2)
fig.tight_layout(); fig.savefig(OUT/'s13_r01_error_changes.png', dpi=160); plt.show()
# To compare any two completed experiments, call compare_experiments with their IDs.

## 6. Original model code: slice tokens → twelve outputs

다음 정의는 우리 `experiments/spatial_core.py`에서 그대로 추출한 원본입니다. 이미지 encoder 다음의 head이며, MRI 전체 학습 파이프라인을 대체하지 않습니다. 실제 학습에서는 training fold의 feature mean/std와 유효 slice mask를 전달합니다. `selected_windows`는 이웃3/5/10과 중복 제거의 차이를 보여 줍니다.

실행 예제는 random feature/target을 사용하는 **shape·gradient 확인용**입니다. 의료 예측이나 성능 측정 결과가 아닙니다.

In [ ]:
def selected_windows(length, config):
    if length < 1 or config['neighbors'] not in (3, 5, 10) or config['centers'] < 1:
        raise ValueError('Invalid slice selection.')
    start, end = config.get('center_range', [.25, .75])
    if not 0 <= start < end <= 1: raise ValueError('Invalid center range.')
    centers = np.unique(np.rint(np.linspace(start, end, config['centers']) * (length-1)).astype(int))
    if config['representation'] == 'triplet':
        if config['neighbors'] != 3:
            raise ValueError('Triplet encoder requires three channels.')
        return [tuple(np.clip([c-1,c,c+1], 0, length-1)) for c in centers]
    if config['representation'] != 'single':
        raise ValueError('Unknown representation.')
    left = (config['neighbors'] - 1) // 2
    indices = {int(np.clip(c+j, 0, length-1)) for c in centers
               for j in range(-left, config['neighbors']-left)}
    return [(i,) for i in sorted(indices)]

def rotary(x, position):
    dim = x.shape[-1]
    if dim % 2:
        raise ValueError('Rotary head dimension must be even.')
    frequencies = torch.exp(torch.arange(0, dim, 2, device=x.device, dtype=torch.float32) * (-math.log(10000.) / dim))
    angle = position.float()[:,None,:,None] * frequencies
    a, b = x.float()[...,0::2], x.float()[...,1::2]
    return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), dim=-1).flatten(-2).to(x.dtype)

class SliceBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv, self.out = nn.Linear(dim, 3*dim), nn.Linear(dim, dim)
        self.ffn = nn.Sequential(nn.Linear(dim,2*dim),nn.GELU(),nn.Linear(2*dim,dim))

    def forward(self, x, mask, position, use_rope):
        b, t, d = x.shape
        q,k,v = self.qkv(self.norm1(x)).reshape(b,t,3,self.heads,d//self.heads).permute(2,0,3,1,4).unbind(0)
        if use_rope:
            q,k = rotary(q,position),rotary(k,position)
        scores = (q.float() @ k.float().transpose(-1,-2)) / math.sqrt(d//self.heads)
        scores = scores.masked_fill(~mask[:,None,None,:], -1e4).softmax(-1)
        context = (scores.to(v.dtype) @ v).transpose(1,2).reshape(b,t,d)
        x = x + self.out(context)
        return (x + self.ffn(self.norm2(x))) * mask[...,None]

class SpatialHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.config = config
        d = config['hidden_dim']
        if d % config['heads'] or (d//config['heads']) % 2:
            raise ValueError('Invalid attention dimensions.')
        self.register_buffer('mean', mean.float().clone())
        self.register_buffer('std', std.float().clone())
        self.project = nn.Linear(config['feature_dim'], d)
        self.pool_score = nn.Sequential(nn.Linear(d,64),nn.Tanh(),nn.Linear(64,12))
        self.class_weight = nn.Parameter(torch.empty(12,6*d))
        self.class_bias = nn.Parameter(torch.zeros(12))
        nn.init.normal_(self.class_weight,std=.01)
        self.coord_kind = nn.Embedding(2,d)
        nn.init.zeros_(self.coord_kind.weight)
        if config['sequence']:
            self.slice_block = SliceBlock(d,config['heads'])
        if config['plane_embedding']:
            self.plane = nn.Embedding(3,d)
            self.protocol = nn.Embedding(2,d)
        mode = config['fusion']
        if mode == 'mlp':
            self.mlp = nn.Sequential(nn.Linear(6*d,d),nn.GELU(),nn.Linear(d,1))
        elif mode == 'gate':
            self.gate = nn.Linear(d,1)
            self.gate_weight = nn.Parameter(torch.randn(12,d)*.01)
        elif mode == 'cross':
            self.queries = nn.Parameter(torch.randn(12,d)*.02)
            self.memory_norm, self.query_norm = nn.LayerNorm(d),nn.LayerNorm(d)
            self.cross = nn.MultiheadAttention(d,config['heads'],batch_first=True,dropout=0.)
            self.read_norm = nn.LayerNorm(d)
            self.read_ffn = nn.Sequential(nn.Linear(d,2*d),nn.GELU(),nn.Linear(2*d,d))
            self.read_weight = nn.Parameter(torch.randn(12,d)*.01)
        elif mode != 'concat':
            raise ValueError('Unsupported fusion.')

    def forward(self, tokens, mask, coords):
        if tokens.ndim != 4 or tokens.shape[1] != 6 or mask.shape != tokens.shape[:3] or coords.shape != (*mask.shape,3):
            raise ValueError('Invalid token/coordinate shape.')
        if not mask.flatten(1).any(1).all():
            raise ValueError('Each study requires a valid image.')
        coords = torch.where(mask[...,None],coords,0.)
        clean = torch.where(mask[...,None],tokens.float(),self.mean)
        if not torch.isfinite(coords).all() or not torch.isfinite(clean).all():
            raise ValueError('Non-finite valid token or coordinate.')
        x = self.project((clean-self.mean)/self.std)
        x = x + self.coord_kind((coords[...,2]>.5).long())
        b,s,t,d = x.shape
        if self.config['plane_embedding']:
            slot=torch.arange(6,device=x.device)
            x=x+(self.plane(slot//2)+self.protocol(slot%2))[None,:,None,:]
        pos=coords[...,0]
        if self.config['position']=='rope_mm':
            pos=torch.where(coords[...,2]>.5,coords[...,1]/self.config['mm_unit'],pos)
        if self.config['position']=='additive':
            freq=torch.exp(torch.arange(0,d,2,device=x.device)*(-math.log(10000.)/d))
            angles=pos[...,None]*freq
            x=x+torch.stack((angles.sin(),angles.cos()),-1).flatten(-2)
        x=x*mask[...,None]
        if self.config['sequence']:
            flat=x.reshape(b*s,t,d); flat_mask=mask.reshape(b*s,t); active=flat_mask.any(1)
            result=torch.zeros_like(flat)
            result[active]=self.slice_block(flat[active],flat_mask[active],pos.reshape(b*s,t)[active],self.config['position'].startswith('rope'))
            x=result.reshape(b,s,t,d)
        if self.config['fusion']=='cross':
            memory=self.memory_norm(x.flatten(1,2)); query=self.queries[None].expand(b,-1,-1)
            read,_=self.cross(self.query_norm(query),memory,memory,key_padding_mask=~mask.flatten(1),need_weights=False)
            read=query+read; read=read+self.read_ffn(self.read_norm(read))
            return (read*self.read_weight).sum(-1)+self.class_bias
        attention=self.pool_score(x).masked_fill(~mask[...,None],-1e4).softmax(2)*mask[...,None]
        pooled=torch.einsum('bstc,bstd->bscd',attention,x).permute(0,2,1,3)
        if self.config['fusion']=='mlp':
            return self.mlp(pooled.flatten(2)).squeeze(-1)+self.class_bias
        if self.config['fusion']=='gate':
            weight=self.gate(pooled).squeeze(-1).masked_fill(~mask.any(2)[:,None,:],-1e4).softmax(-1)
            return ((pooled*weight[...,None]).sum(2)*self.gate_weight).sum(-1)+self.class_bias
        return (pooled.flatten(2)*self.class_weight).sum(-1)+self.class_bias

In [ ]:
torch.manual_seed(42)
demo_config = dict(feature_dim=768, hidden_dim=128, heads=4, sequence=True,
                   plane_embedding=True, position='rope_mm', mm_unit=3.0, fusion='cross')
model = SpatialHead(demo_config, torch.zeros(768), torch.ones(768))
B,S,T,D = 2,6,5,768
tokens = torch.randn(B,S,T,D)
mask = torch.ones(B,S,T,dtype=torch.bool)
mask[1,5] = False  # One missing series; padded tokens must not participate.
coords = torch.zeros(B,S,T,3)
coords[...,0] = torch.arange(T)
coords[...,1] = torch.arange(T) * 3.0
coords[...,2] = 1.0  # Physical DICOM coordinates available.
logits = model(tokens,mask,coords)
synthetic_targets = torch.rand(B,12)
synthetic_loss = nn.functional.binary_cross_entropy_with_logits(logits,synthetic_targets)
synthetic_loss.backward()
assert logits.shape == (B,12) and torch.isfinite(logits).all()
assert model.project.weight.grad is not None
assert torch.isfinite(model.project.weight.grad).all()
print('Synthetic shape check:',tuple(tokens.shape),'->',tuple(logits.shape))
print('BCE backward succeeded; these random scores have no performance meaning.')
for neighbours in (3,5,10):
    cfg = dict(centers=5, neighbors=neighbours, representation='single')
    picked = selected_windows(30,cfg)
    print('Neighbours',neighbours,':',len(picked),'unique tokens from a 30-slice series')

## 7. What public high-score notebooks add

We inspected **25 original source snapshots: 22 high-score candidates/derivatives and 3 training or architecture references**. This is not a claim that 22 independently reproduced 0.95 solutions were verified. Some current versions are unscored, reuse another author's score, require private extra weights, or only launch external code. Public figures below remain author reports.

1. **Input geometry and coverage:** the compact R7 manifest uses ConvNeXt-small, render384/crop352, physical spacing and knee-centred views. Raptor's commonly named 384 branch actually takes a320 crop from a384 rendered image and evaluates94 triplets over a96-slice stack. Copying a model name alone misses this contract.
2. **Image adaptation:** R7 records16epochs and4folds; the original Raptor training code adapts backbone+head with report soft targets. Our search mostly trained heads on frozen Swin features, with a limited one-epoch adaptation experiment.
3. **Pooling before information is lost:** R7's active spatial attention works on the image feature map before class-specific window pooling. Our cross-attention starts after image global pooling. This is a meaningful next ablation, not evidence that all Transformer heads must win.
4. **Complementarity:** the224 and320 Raptor branches use distinct trained weights and fields of view. Their rank blend is different from merely resizing one network's input. Anatomical mirror TTA must respect plane/channel order.
5. **Negative evidence matters:** goodpjw reports adding S6 reduced .958→.957; tuning12weights onGold58 reduced the reported LB to.955. hitarthjain0 reports a weak extra model lowered.950→.948. More members and more tuning do not guarantee improvement.
6. **Read execution, not labels:** several notebooks are the same inference family under different titles. We also found a four-model candidate explicitly marked unmeasured and a public.957 article without executable inference in that source version.

For AUC, a strictly increasing post-transform preserves order. Dividing a final score by a positive temperature alone cannot improve AUC. Rank or probit **before mixing models** can change the order, but neither is universally superior. Correlation-based clinical score adjustments also need validation; plausible stories are not causal evidence.

### Source ledger

| # | Source snapshot | Version |
|---|---|---|
| 1 | [goodpjw2008/rsna-knee-coatnet-r6-2-5d-mil-lb-0-958](https://www.kaggle.com/code/goodpjw2008/rsna-knee-coatnet-r6-2-5d-mil-lb-0-958) | 27 |
| 2 | [goodpjw2008/rsna-knee-does-s6-help-next-to-r6-lb-0-957](https://www.kaggle.com/code/goodpjw2008/rsna-knee-does-s6-help-next-to-r6-lb-0-957) | 8 |
| 3 | [heliosli/this-is-best-public-dataset-new-version-0-9586](https://www.kaggle.com/code/heliosli/this-is-best-public-dataset-new-version-0-9586) | 7 |
| 4 | [heliosli/this-one-is-old-version-0-9573-newone-you-can-use](https://www.kaggle.com/code/heliosli/this-one-is-old-version-0-9573-newone-you-can-use) | 2 |
| 5 | [heliosli/blend-mine-in-private-lb-luck-to-all-upvoters](https://www.kaggle.com/code/heliosli/blend-mine-in-private-lb-luck-to-all-upvoters) | 1 |
| 6 | [heliosli/0-957-latest-dataset-score-in-one-click](https://www.kaggle.com/code/heliosli/0-957-latest-dataset-score-in-one-click) | 1 |
| 7 | [evgendvorkin/rsna-versia-7](https://www.kaggle.com/code/evgendvorkin/rsna-versia-7) | 22 |
| 8 | [prvsiyan/the-bee-s-knees-final-rsna-push](https://www.kaggle.com/code/prvsiyan/the-bee-s-knees-final-rsna-push) | 47 |
| 9 | [kaiwalyaatulraut/rsna-knee-competition-solution](https://www.kaggle.com/code/kaiwalyaatulraut/rsna-knee-competition-solution) | 3 |
| 10 | [kiyoshiohno/rsna-knee-template](https://www.kaggle.com/code/kiyoshiohno/rsna-knee-template) | 27 |
| 11 | [bliverrigpdoge/r7-raptor-knee-centred-convnext](https://www.kaggle.com/code/bliverrigpdoge/r7-raptor-knee-centred-convnext) | 1 |
| 12 | [laimerbotrolenk/rsna-four-model-ensemble](https://www.kaggle.com/code/laimerbotrolenk/rsna-four-model-ensemble) | 1 |
| 13 | [leoprovorov/rsna-knee-0-957-explained-why-the-open-ensembl](https://www.kaggle.com/code/leoprovorov/rsna-knee-0-957-explained-why-the-open-ensembl) | 12 |
| 14 | [jenillangaliya/rsna-knee-0-956-raptor-convnext-r7-ensemble](https://www.kaggle.com/code/jenillangaliya/rsna-knee-0-956-raptor-convnext-r7-ensemble) | 1 |
| 15 | [jenillangaliya/rsna-knee-lb-0-954-dual-sota-ensemble](https://www.kaggle.com/code/jenillangaliya/rsna-knee-lb-0-954-dual-sota-ensemble) | 4 |
| 16 | [varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956](https://www.kaggle.com/code/varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956) | 1 |
| 17 | [varshithakolupuri/rsna-0-956-one-free-flip-68-ranks-up](https://www.kaggle.com/code/varshithakolupuri/rsna-0-956-one-free-flip-68-ranks-up) | 1 |
| 18 | [hitarthjain0/rsna-knee-coatnet-convnext-mil-lb-0-950](https://www.kaggle.com/code/hitarthjain0/rsna-knee-coatnet-convnext-mil-lb-0-950) | 5 |
| 19 | [rabari9999/rsna-knee-0-956-micro-boost-clinical-tri-reader](https://www.kaggle.com/code/rabari9999/rsna-knee-0-956-micro-boost-clinical-tri-reader) | 11 |
| 20 | [yuvrajlamba01/rsna-knee-multiscale-orthofoundation-sota-0-956](https://www.kaggle.com/code/yuvrajlamba01/rsna-knee-multiscale-orthofoundation-sota-0-956) | 2 |
| 21 | [rabari9999/rsna-knee-sota-dual-engine-stack-lb-0-951](https://www.kaggle.com/code/rabari9999/rsna-knee-sota-dual-engine-stack-lb-0-951) | 4 |
| 22 | [yuvrajlamba01/rsna-knee-v15-tri-hybrid-ensemble-0-954](https://www.kaggle.com/code/yuvrajlamba01/rsna-knee-v15-tri-hybrid-ensemble-0-954) | 8 |
| 23 | [pilkwang/rsna-knee-baseline-v1](https://www.kaggle.com/code/pilkwang/rsna-knee-baseline-v1) | 15 |
| 24 | [dreaddevelopment/knee-mri-training-the-twelve-finding-model](https://www.kaggle.com/code/dreaddevelopment/knee-mri-training-the-twelve-finding-model) | 8 |
| 25 | [mattiaangeli/bend-the-knee-to-speedy-raptors-the-original](https://www.kaggle.com/code/mattiaangeli/bend-the-knee-to-speedy-raptors-the-original) | 39 |

Model configuration evidence: [R7 manifest dataset](https://www.kaggle.com/datasets/heliosli/rsna-knee-compact-r7cnxfsss). These links may later show newer versions; the table records the source versions inspected on2026-10-19. Original code/model licenses and dataset conditions remain with their authors. The source review was static; no third-party checkpoint or embedded script was executed for this journal.

## 8. Next experiments — hypotheses, not completed results

현재 별도 자동 queue에는 confidence weighting, 소견별 loss 균형, 중심 범위 확대, 중심 수 증가, head warmup 후 backbone fine-tuning 비교가 있습니다. 이 노트의 완료 결과표에는 포함하지 않았습니다.

| Priority | Controlled change | Evidence to collect |
|---|---|---|
| P00 | R01 probability blend vs fixed-weight rank blend | Same-fold OOF AUC/AP, error correlation, uncertainty; retain probabilities for calibration |
| P01 | Physical mm crop, then laterality normalization in separate steps | Header coverage, image contract, medial/lateral category errors |
| P02 | Broad coverage vs dense neighbouring slices, same unique-token budget | Coverage audit and paired OOF comparison |
| P03 | Global pooling vs class-specific spatial patch pooling | Same encoder/input/training data; memory/time and category AUC |
| P04 | Warmup then last-stage1/3/6epoch adaptation | Training curves, three seeds, deterioration on weak categories |
| P05 | Best checkpoint vs predeclared EMA/weight averaging | Same training run, fixed checkpoint-selection rule |
| P06 | Plain vs geometry-consistent mirror TTA | Same weights, OOF scores, inference cost |
| P07 | Backbone comparison after fixing input/training conditions | Swin/ConvNeXt/CoAtNet under matched budget |

채택은 평균 AUC 한 숫자만으로 결정하지 않습니다. seed 일관성, MCL/Fracture 등 취약 소견, paired uncertainty, 계산 비용을 함께 기록합니다. 58gold에 소견별 비율을 많이 맞추는 실험은 별도의 inner validation/독립 holdout 설계가 필요합니다. 외부 모델이 같은 gold로 선택되었다면 그 출력은 새로운 clean OOF라고 부르지 않습니다.

이 프로젝트에서 얻는 경험은 모델 이름을 바꾸는 것보다 **의료영상의 좌표·입력 계약, 약한 감독의 불확실성, 통제된 실험, 오류 분석, 재현 가능한 학습/추론 자동화**를 연결하는 데 있습니다. 대회 성능이 실제 진료 유효성을 검증하는 것은 아닙니다. 새 결과가 검증되면 이 노트를 버전별로 업데이트할 계획입니다.

In [ ]:
summary = {'snapshot':'2026-10-19','completed_experiments':len(stats),
           'evaluation':'Gold58 development 3-fold OOF, three training seeds',
           'best_experiment':'r01_e09_s13_blend','mean_auc':float(stats.set_index('experiment').loc['r01_e09_s13_blend','macro_auc']),
           'official_leaderboard_score_claimed':False,
           'public_sources_inspected':25,'patient_level_records_included':False}
(OUT/'journal_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))